# Donor Behavioral Clustering — EFA-Informed v2

A deliberately simple clustering notebook built from the factor-analysis findings.

### Default geometry
K-means sees **one standardized score per behavioral unit**, not the raw component features.

1. Transform / coverage-rule / median-impute raw components.
2. Standardize each raw component.
3. Flip signs where needed so every component points in the unit's named direction.
4. Average components **equally within each unit**.
5. Standardize each unit score.
6. Run plain K-means on the unit-score matrix.

**No factor-loading weights, stakeholder weights, PCA, or feature-count weighting.**

The default core uses four units:
- relationship loyalty
- giving rhythm
- amount consistency
- funding posture

The earlier feature groups are retained in an optional-unit library for quick experiments. `campaign_responsiveness` is the first add-on I would test.

**Window note:** the current source build uses explicit `_24m` field names.

In [1]:
from __future__ import annotations

from pathlib import Path
from html import escape
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print(f"pandas={pd.__version__} | numpy={np.__version__}")

pandas=2.3.2 | numpy=1.26.4


## 1 — Configuration

For a baseline run, edit only:
- `MIN_GIFTS_FIELD` / `MIN_GIFTS`
- `EXCLUDE_TEACHERS`
- `SCORE_ALL_REPEAT`
- `ACTIVE_OPTIONAL_UNITS`
- `SELECTED_K`

Examples:

```python
ACTIVE_OPTIONAL_UNITS = []                            # four-unit core
ACTIVE_OPTIONAL_UNITS = ["campaign_responsiveness"]  # preferred first add-on
ACTIVE_OPTIONAL_UNITS = ["jtbd_values_equity"]       # older JTBD sensitivity
```

The notebook blocks any active configuration that reuses the same raw feature in two units, preventing accidental double-counting.

In [2]:
DATA_DIR = Path("/Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data")
FEATURES_PATH = DATA_DIR / "clustering_features_20260801_w24m.csv"

# Population controls.
MIN_GIFTS = 4
MIN_GIFTS_FIELD = "n_gifts_green_24m"   # use "n_gifts_24m" for all project gifts
TOTAL_GIFTS_FIELD = "n_gifts_24m"
EXCLUDE_TEACHERS = False
SCORE_ALL_REPEAT = False   # after fitting, score donors with 2 <= MIN_GIFTS_FIELD < MIN_GIFTS

# K-means controls.
K_VALUES = [3, 4, 5, 6, 7, 8]
SELECTED_K = 6
RANDOM_STATE = 20260825
N_INIT = 20
SILHOUETTE_SAMPLE_N = 5000
STABILITY_RUNS = 10
STABILITY_SAMPLE_FRAC = 0.80

PROJECT_COST_COVERAGE_FIELD = "coverage_project_cost_24m"
PROJECT_COST_COVERAGE_FLOOR = 0.50

# -------------------------------------------------------------------
# CORE: EFA-informed units.
# {feature: direction}; direction is only a sign flip.
# Every component receives equal weight within its unit.
# -------------------------------------------------------------------
ACTIVE_OPTIONAL_UNITS = ["trigger", "jtbd_local_stewardship", "choice_breadth"]

# Relative influence in K-means squared-distance geometry.
# Unlisted units default to 1.00.
UNIT_WEIGHTS = {
    "jtbd_local_stewardship": 0.5,
    "choice_breadth": 0.5,
}

CORE_UNITS = {
    "relationship_loyalty": {
        "share_gifts_repeat_teacher_24m": +1,
        "entropy_school_norm_24m": -1,
    },
    "giving_rhythm": {
        "entropy_gift_month_norm_24m": +1,
        "gifts_per_active_month_24m": -1,
    },
    "giving_approach": {
        "modal_amount_share_24m": +1,
        "share_gifts_round_amount_24m": +1,
        "median_gift_to_project_cost_ratio_24m": -1,
        "share_gifts_closed_project_24m": -1,
        "share_gifts_first_money_in_24m": +1,
    },
}

# -------------------------------------------------------------------
# OPTIONAL LIBRARY.
# Keeps the prior clustering/JTBD groups available for testing.
# Add one idea at a time when possible.
# -------------------------------------------------------------------
OPTIONAL_UNIT_LIBRARY = {
    "campaign_responsiveness": {
        "share_gifts_with_match_24m": +1,
        "share_gifts_big_event_24m": +1,
    },
    "choice_breadth": {
        "top_category_share_count_24m": -1,
        "entropy_category_24m": +1,

    },
    "seasonality": {
        "top_month_share_24m": +1,
        "share_gifts_q4_24m": +1,
    },
    "trigger": {
        "share_gifts_with_match_24m": +1,
        "share_gifts_big_event_24m": +1,
    },
    "funding_depth": {
        "share_gifts_over_half_project_cost_24m": +1,
        "share_gifts_full_project_cost_24m": +1,
    },
    "jtbd_decision_deliberation": {
        "share_gifts_with_prior_search_24m": +1,
        "project_page_pre_gift_mean_24m": +1,
    },
    "jtbd_giving_leverage": {
        "share_gifts_with_match_24m": +1,
        "mean_match_excess_24m": +1,
    },
    "jtbd_sustained_monthly": {
        "is_monthly_donor_current": +1,
        "monthly_longest_streak_months": +1,
    },
    "jtbd_tax_efficiency": {
        "share_gifts_daf_24m": +1,
    },
    "jtbd_active_participation": {
        "sharing_events_24m": +1,
        "sharing_active_months_24m": +1,
    },
    "jtbd_local_stewardship": {
        "share_gifts_within_15mi_24m": +1,
    },
    "jtbd_recognition_avoidance": {
        "share_gifts_anonymous_24m": +1,
    },
    "jtbd_values_equity": {
        "share_gifts_to_low_income_schools_24m": +1,
        "share_gifts_to_historically_underrepresented_race_schools_24m": +1,
    },
    # Extra EFA-informed sensitivities worth keeping nearby.
    "site_engagement": {
        "days_with_site_activity_24m": +1,
        "search_visits_day_total_24m": +1,
    },
    "platform_support": {
        "avg_optional_donation_rate_24m": +1,
        "share_gifts_with_optional_donation_24m": +1,
    },
}

LOG1P_FIELDS = {
    "gifts_per_active_month_24m",
    "median_gift_to_project_cost_ratio_24m",
    "project_page_pre_gift_mean_24m",
    "mean_match_excess_24m",
    "sharing_events_24m",
    "monthly_longest_streak_months",
    "days_with_site_activity_24m",
    "search_visits_day_total_24m",
}

PROJECT_COST_FIELDS = {
    "median_gift_to_project_cost_ratio_24m",
    "share_gifts_over_half_project_cost_24m",
    "share_gifts_full_project_cost_24m",
}

PROFILE_FIELDS = [
    # scale / value
    "n_gifts_24m", "n_gifts_green_24m", "gift_amount_24m", "median_gift_amount_24m",
    "lifetime_amount", "lifetime_gift_count", "is_major_gift_donor",
    # lifecycle / cadence
    "n_active_months_24m", "days_since_last_gift", "tenure_days",
    "is_new_donor_24m", "is_reactivated_24m", "is_continuing_donor_24m",
    # monthly
    "is_monthly_donor_current", "monthly_active_months_24m",
    "monthly_longest_streak_months", "share_amount_monthly_24m",
    # timing / campaign
    "share_gifts_year_end_final_week_24m", "share_gifts_giving_tuesday_window_24m",
    "share_gifts_back_to_school_24m", "share_gifts_summer_24m", "share_gifts_q4_24m",
    "top_month_share_24m", "share_gifts_with_match_24m", "mean_match_excess_24m",
    "share_gifts_big_event_24m",
    # product / choice / relationship
    "share_gifts_classroom_essentials_24m", "entropy_category_norm_24m",
    "n_unique_categories_24m", "top_category_share_count_24m",
    "n_unique_schools_24m", "share_gifts_repeat_school_24m",
    # funding context
    "median_project_total_cost_24m", "share_gifts_over_half_project_cost_24m",
    "share_gifts_full_project_cost_24m",
    # geography / equity
    "median_distance_mi_24m", "share_gifts_within_15mi_24m",
    "share_gifts_to_low_income_schools_24m",
    "share_gifts_to_historically_underrepresented_race_schools_24m",
    "share_gifts_to_underserved_rural_schools_24m",
    # donor / engagement context
    "is_teacher", "is_teacher_referred", "is_marketing_subscribed",
    "email_open_rate_24m", "email_click_rate_24m",
    "days_with_site_activity_24m", "n_site_visits_24m", "search_visits_day_total_24m",
    "sharing_events_24m", "sharing_active_months_24m",
    "share_gifts_daf_24m", "share_gifts_anonymous_24m", "share_gifts_green_24m",
    # platform support
    "avg_optional_donation_rate_24m", "share_gifts_with_optional_donation_24m",
]

PROFILE_TOP_N = 8
PROFILE_MIN_COVERAGE = 0.50
DEMO_IF_FEATURE_FILE_MISSING = True

assert SELECTED_K in K_VALUES
for name in ACTIVE_OPTIONAL_UNITS:
    if name not in OPTIONAL_UNIT_LIBRARY:
        raise ValueError(f"Unknown optional unit: {name}")

print("Active optional units:", ACTIVE_OPTIONAL_UNITS or "none - four-unit core")

Active optional units: ['trigger', 'jtbd_local_stewardship', 'choice_breadth']


## 2 — Load the donor matrix

Active clustering fields are strict. Inactive optional-library fields and profile fields are loaded when available but do not make the baseline notebook fail.

In [3]:
def active_units(optional_units=None):
    optional_units = ACTIVE_OPTIONAL_UNITS if optional_units is None else list(optional_units)
    units = {name: dict(spec) for name, spec in CORE_UNITS.items()}

    for name in optional_units:
        if name not in OPTIONAL_UNIT_LIBRARY:
            raise ValueError(f"Unknown optional unit: {name}")
        units[name] = dict(OPTIONAL_UNIT_LIBRARY[name])

    seen = {}
    duplicates = {}
    for unit, spec in units.items():
        for feature in spec:
            if feature in seen:
                duplicates.setdefault(feature, [seen[feature]]).append(unit)
            else:
                seen[feature] = unit

    if duplicates:
        msg = "; ".join(f"{f}: {u}" for f, u in duplicates.items())
        raise ValueError(f"Active units reuse the same raw feature. Choose one version of the construct: {msg}")

    return units


def flatten_unit_features(units):
    return [feature for spec in units.values() for feature in spec]


def all_library_features():
    fields = set(flatten_unit_features(CORE_UNITS))
    for spec in OPTIONAL_UNIT_LIBRARY.values():
        fields.update(spec)
    return sorted(fields)


def make_demo_data(n=3500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    latent = rng.choice(5, size=n, p=[0.24, 0.22, 0.20, 0.19, 0.15])

    d = pd.DataFrame({
        "donor_id": [f"demo_{i:05d}" for i in range(n)],
        "cluster_eligible": 1,
        "is_teacher": (rng.random(n) < 0.12).astype(float),
        PROJECT_COST_COVERAGE_FIELD: np.clip(rng.normal(0.90, 0.10, n), 0, 1),
    })

    total_gifts = np.maximum(4, np.rint(np.exp(1.6 + 0.22 * latent + rng.normal(0, 0.35, n))).astype(int))
    green_share = np.clip(0.55 + 0.06 * latent + rng.normal(0, 0.12, n), 0.10, 1.0)
    d["n_gifts_24m"] = total_gifts
    d["n_gifts_green_24m"] = np.rint(total_gifts * green_share).astype(int)

    loyalty = (latent - latent.mean()) / max(latent.std(), 1e-9)
    d["share_gifts_repeat_teacher_24m"] = np.clip(0.55 + 0.14 * loyalty + rng.normal(0, 0.12, n), 0, 1)
    d["entropy_school_norm_24m"] = np.clip(0.55 - 0.15 * loyalty + rng.normal(0, 0.12, n), 0, 1)

    rhythm = rng.normal(size=n)
    d["entropy_gift_month_norm_24m"] = np.clip(0.55 + 0.18 * rhythm + rng.normal(0, 0.10, n), 0, 1)
    d["gifts_per_active_month_24m"] = np.clip(2.7 - 0.70 * rhythm + rng.normal(0, 0.35, n), 1, 7)

    consistency = rng.normal(size=n)
    d["modal_amount_share_24m"] = np.clip(0.55 + 0.18 * consistency + rng.normal(0, 0.10, n), 0, 1)
    d["share_gifts_round_amount_24m"] = np.clip(0.55 + 0.17 * consistency + rng.normal(0, 0.10, n), 0, 1)

    posture = rng.normal(size=n)
    d["median_gift_to_project_cost_ratio_24m"] = np.exp(-2.0 + 0.55 * posture + rng.normal(0, 0.35, n))
    d["share_gifts_closed_project_24m"] = np.clip(0.30 + 0.16 * posture + rng.normal(0, 0.12, n), 0, 1)
    d["share_gifts_first_money_in_24m"] = np.clip(0.45 - 0.14 * posture + rng.normal(0, 0.12, n), 0, 1)

    requested = set(all_library_features()) | set(PROFILE_FIELDS)
    requested |= {MIN_GIFTS_FIELD, TOTAL_GIFTS_FIELD, PROJECT_COST_COVERAGE_FIELD}

    for f in requested:
        if f in d.columns:
            continue
        name = f.lower()
        if name.startswith(("share_", "is_", "has_")) or "_rate" in name or "entropy_" in name:
            d[f] = rng.beta(2, 4, n)
        elif "amount" in name and "share_" not in name and "_rate" not in name:
            d[f] = np.exp(rng.normal(4.0, 0.7, n))
        elif "ratio" in name or "match_excess" in name:
            d[f] = np.exp(rng.normal(-1.5, 0.6, n))
        elif any(token in name for token in ["days", "months", "count", "n_unique", "visits", "events", "payments"]):
            d[f] = np.maximum(0, np.rint(np.exp(rng.normal(1.5, 0.6, n))))
        else:
            d[f] = rng.normal(0, 1, n)

    for f, rate in {
        "median_gift_to_project_cost_ratio_24m": 0.06,
        "median_distance_mi_24m": 0.30,
        "share_gifts_within_15mi_24m": 0.30,
    }.items():
        if f in d.columns:
            d.loc[rng.random(n) < rate, f] = np.nan

    return d


UNITS_NOW = active_units()
FIT_FIELDS_NOW = flatten_unit_features(UNITS_NOW)

control_fields = [
    "donor_id", "cluster_eligible", MIN_GIFTS_FIELD, TOTAL_GIFTS_FIELD,
    "is_teacher", PROJECT_COST_COVERAGE_FIELD,
]
candidate_fields = list(dict.fromkeys(
    control_fields + FIT_FIELDS_NOW + all_library_features() + PROFILE_FIELDS
))

if FEATURES_PATH.exists():
    header = pd.read_csv(FEATURES_PATH, nrows=0).columns.tolist()
    header_set = set(header)

    required = ["donor_id", "cluster_eligible", MIN_GIFTS_FIELD] + FIT_FIELDS_NOW
    if EXCLUDE_TEACHERS:
        required.append("is_teacher")

    missing_required = [f for f in required if f not in header_set]
    if missing_required:
        raise ValueError(
            "Required active fields missing from feature CSV:\n"
            + "\n".join(f"  - {f}" for f in missing_required)
        )

    usecols = [f for f in candidate_fields if f in header_set]
    FEATURES = pd.read_csv(FEATURES_PATH, usecols=usecols)
    DATA_SOURCE = str(FEATURES_PATH)
else:
    if not DEMO_IF_FEATURE_FILE_MISSING:
        raise FileNotFoundError(FEATURES_PATH)
    FEATURES = make_demo_data()
    DATA_SOURCE = "SYNTHETIC SMOKE TEST - real donor-level feature CSV is not attached here"

if FEATURES["donor_id"].duplicated().any():
    raise ValueError("donor_id must be unique")

eligibility = pd.to_numeric(FEATURES[MIN_GIFTS_FIELD], errors="coerce")
mask = pd.to_numeric(FEATURES["cluster_eligible"], errors="coerce").eq(1) & eligibility.ge(MIN_GIFTS)

if EXCLUDE_TEACHERS:
    mask &= ~pd.to_numeric(FEATURES["is_teacher"], errors="coerce").eq(1)

DONORS = FEATURES.loc[mask].copy().set_index("donor_id")

if len(DONORS) <= SELECTED_K:
    raise ValueError(f"Only {len(DONORS)} eligible donors; not enough for K={SELECTED_K}")

print(f"Data source: {DATA_SOURCE}")
print(f"Eligible donors: {len(DONORS):,} | {MIN_GIFTS_FIELD} >= {MIN_GIFTS} | EXCLUDE_TEACHERS={EXCLUDE_TEACHERS}")
print(f"Active units ({len(UNITS_NOW)}): {list(UNITS_NOW)}")
print(f"Raw component fields ({len(FIT_FIELDS_NOW)}): {FIT_FIELDS_NOW}")

Data source: /Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data/clustering_features_20260801_w24m.csv
Eligible donors: 54,631 | n_gifts_green_24m >= 4 | EXCLUDE_TEACHERS=False
Active units (6): ['relationship_loyalty', 'giving_rhythm', 'giving_approach', 'trigger', 'jtbd_local_stewardship', 'choice_breadth']
Raw component fields (14): ['share_gifts_repeat_teacher_24m', 'entropy_school_norm_24m', 'entropy_gift_month_norm_24m', 'gifts_per_active_month_24m', 'modal_amount_share_24m', 'share_gifts_round_amount_24m', 'median_gift_to_project_cost_ratio_24m', 'share_gifts_closed_project_24m', 'share_gifts_first_money_in_24m', 'share_gifts_with_match_24m', 'share_gifts_big_event_24m', 'share_gifts_within_15mi_24m', 'top_category_share_count_24m', 'entropy_category_24m']


## 3 — Sanity-check the active raw components

This is only a guardrail for missing, degenerate, or extremely discrete component features before they are collapsed into unit scores.

In [4]:
def feature_sanity_table(df, units):
    feature_to_unit = {f: unit for unit, spec in units.items() for f in spec}
    direction_lookup = {f: direction for spec in units.values() for f, direction in spec.items()}
    rows = []

    for f in flatten_unit_features(units):
        x = pd.to_numeric(df[f], errors="coerce")
        observed = x.dropna()
        rows.append({
            "unit": feature_to_unit[f],
            "feature": f,
            "direction": "+" if direction_lookup[f] > 0 else "-",
            "missing": x.isna().mean(),
            "n_unique": observed.nunique(),
            "std": observed.std(ddof=0),
            "p10": observed.quantile(0.10) if len(observed) else np.nan,
            "p50": observed.quantile(0.50) if len(observed) else np.nan,
            "p90": observed.quantile(0.90) if len(observed) else np.nan,
            "% zero": (observed == 0).mean() if len(observed) else np.nan,
            "% one": (observed == 1).mean() if len(observed) else np.nan,
            "transform": "log1p" if f in LOG1P_FIELDS else "none",
        })

    return pd.DataFrame(rows)


SANITY = feature_sanity_table(DONORS, UNITS_NOW)
display(
    SANITY.style
    .format({
        "missing": "{:.1%}", "std": "{:.3f}", "p10": "{:.3f}",
        "p50": "{:.3f}", "p90": "{:.3f}", "% zero": "{:.1%}", "% one": "{:.1%}"
    })
    .hide(axis="index")
    .set_caption("Active raw-component sanity check")
)

bad = SANITY.loc[(SANITY["n_unique"] <= 1) | (SANITY["std"].fillna(0) <= 1e-12), "feature"].tolist()
if bad:
    raise ValueError(f"Degenerate active fit features: {bad}")

unit,feature,direction,missing,n_unique,std,p10,p50,p90,% zero,% one,transform
relationship_loyalty,share_gifts_repeat_teacher_24m,+,0.1%,2427,0.327,0.000,0.467,1.000,12.6%,12.3%,none
relationship_loyalty,entropy_school_norm_24m,-,0.7%,6973,0.378,0.000,0.418,1.000,30.6%,6.9%,none
giving_rhythm,entropy_gift_month_norm_24m,+,0.0%,11682,0.245,0.278,0.656,0.896,4.8%,3.3%,none
giving_rhythm,gifts_per_active_month_24m,-,0.0%,1747,4.025,1.167,2.000,5.000,0.0%,8.5%,log1p
giving_approach,modal_amount_share_24m,+,0.0%,2248,0.237,0.125,0.333,0.750,0.0%,3.2%,none
giving_approach,share_gifts_round_amount_24m,+,0.0%,2493,0.325,0.083,0.560,1.000,7.1%,16.2%,none
giving_approach,median_gift_to_project_cost_ratio_24m,-,0.2%,51958,0.198,0.031,0.111,0.479,0.0%,0.6%,log1p
giving_approach,share_gifts_closed_project_24m,-,0.0%,2168,0.199,0.000,0.107,0.433,35.8%,0.5%,none
giving_approach,share_gifts_first_money_in_24m,+,0.0%,1668,0.167,0.000,0.125,0.400,34.7%,0.1%,none
trigger,share_gifts_with_match_24m,+,0.0%,2409,0.276,0.000,0.286,0.750,16.7%,3.3%,none


## 4 — Build one equally weighted score per behavioral unit

**raw feature → transform / coverage rule → median impute → feature z-score → sign flip → equal mean within unit → unit z-score**

K-means only sees the final unit-score matrix.

In [5]:
def prepare_unit_matrix(df, units):
    fields = flatten_unit_features(units)
    X = df[fields].apply(pd.to_numeric, errors="coerce").copy()

    if PROJECT_COST_COVERAGE_FIELD in df.columns:
        cov = pd.to_numeric(df[PROJECT_COST_COVERAGE_FIELD], errors="coerce")
        low_cov = cov < PROJECT_COST_COVERAGE_FLOOR
        for f in PROJECT_COST_FIELDS.intersection(fields):
            X.loc[low_cov, f] = np.nan

    for f in fields:
        if f in LOG1P_FIELDS:
            if (X[f].dropna() < 0).any():
                raise ValueError(f"{f} has negative values but is declared log1p")
            X[f] = np.log1p(X[f])

    missing_before = X.isna().mean()
    medians = X.median(numeric_only=True)
    all_missing = medians[medians.isna()].index.tolist()
    if all_missing:
        raise ValueError(f"Active fit fields are entirely missing after coverage rules: {all_missing}")

    X_imp = X.fillna(medians)

    feature_scaler = StandardScaler()
    feature_z = pd.DataFrame(
        feature_scaler.fit_transform(X_imp),
        index=X_imp.index,
        columns=X_imp.columns,
    )

    unit_raw = pd.DataFrame(index=X_imp.index)
    for unit, spec in units.items():
        signed = pd.DataFrame(
            {f: feature_z[f] * direction for f, direction in spec.items()},
            index=X_imp.index,
        )
        unit_raw[unit] = signed.mean(axis=1)

    unit_scaler = StandardScaler()
    unit_z = pd.DataFrame(
        unit_scaler.fit_transform(unit_raw),
        index=unit_raw.index,
        columns=unit_raw.columns,
    )

    prep = {
        "fields": fields,
        "units": units,
        "medians": medians,
        "missing_before": missing_before,
        "feature_scaler": feature_scaler,
        "unit_scaler": unit_scaler,
        "feature_z": feature_z,
        "unit_raw": unit_raw,
    }
    return unit_z, prep


# Keep an unweighted copy for profiling so "+1 SD local" retains
# exactly the same interpretation as in the equal-weight run.
Z_UNWEIGHTED, PREP = prepare_unit_matrix(DONORS, UNITS_NOW)

# K-means uses squared Euclidean distance, so sqrt(weight) gives
# the intended relative contribution to distance.
Z = Z_UNWEIGHTED.copy()

for unit in Z.columns:
    weight = float(UNIT_WEIGHTS.get(unit, 1.0))

    if weight <= 0:
        raise ValueError(f"Unit weight must be > 0: {unit}={weight}")

    Z[unit] = Z[unit] * np.sqrt(weight)

print(
    "Fit unit weights:",
    {unit: UNIT_WEIGHTS.get(unit, 1.0) for unit in Z.columns}
)

quality = pd.DataFrame({
    "feature": PREP["fields"],
    "missing_before_impute": [PREP["missing_before"][f] for f in PREP["fields"]],
})
display(
    quality.style
    .format({"missing_before_impute": "{:.1%}"})
    .hide(axis="index")
    .set_caption("Fit matrix ready")
)

display(
    Z.corr().style
    .format("{:+.2f}")
    .set_caption("Correlation among final unit scores")
)

print(f"K-means matrix: {Z.shape[0]:,} donors x {Z.shape[1]} equally weighted units")

Fit unit weights: {'relationship_loyalty': 1.0, 'giving_rhythm': 1.0, 'giving_approach': 1.0, 'trigger': 1.0, 'jtbd_local_stewardship': 0.5, 'choice_breadth': 0.5}


feature,missing_before_impute
share_gifts_repeat_teacher_24m,0.1%
entropy_school_norm_24m,0.7%
entropy_gift_month_norm_24m,0.0%
gifts_per_active_month_24m,0.0%
modal_amount_share_24m,0.0%
share_gifts_round_amount_24m,0.0%
median_gift_to_project_cost_ratio_24m,8.2%
share_gifts_closed_project_24m,0.0%
share_gifts_first_money_in_24m,0.0%
share_gifts_with_match_24m,0.0%


,relationship_loyalty,giving_rhythm,giving_approach,trigger,jtbd_local_stewardship,choice_breadth
relationship_loyalty,+1.00,+0.22,+0.33,+0.26,+0.22,-0.21
giving_rhythm,+0.22,+1.00,+0.32,+0.10,+0.08,+0.15
giving_approach,+0.33,+0.32,+1.00,+0.09,+0.18,-0.09
trigger,+0.26,+0.10,+0.09,+1.00,+0.04,+0.08
jtbd_local_stewardship,+0.22,+0.08,+0.18,+0.04,+1.00,-0.05
choice_breadth,-0.21,+0.15,-0.09,+0.08,-0.05,+1.00


K-means matrix: 54,631 donors x 6 equally weighted units


## 5 — K diagnostics

Use silhouette and cluster balance to compare **K within this exact unit space**. They do not choose the strategic winner automatically.

In [6]:
def canonicalize_labels(labels):
    counts = pd.Series(labels).value_counts().sort_values(ascending=False)
    mapping = {old: new for new, old in enumerate(counts.index, start=1)}
    return np.array([mapping[x] for x in labels], dtype=int)


def subsample_stability_ari(
    Z,
    k,
    runs=STABILITY_RUNS,
    sample_frac=STABILITY_SAMPLE_FRAC,
):
    """
    Fit K-means repeatedly on random donor subsamples.
    Each fitted model then assigns every donor.
    Pairwise ARI measures how consistently the donor partition reappears.
    """
    X = Z.to_numpy()
    n = len(X)
    sample_n = max(k + 1, int(n * sample_frac))

    rng = np.random.default_rng(RANDOM_STATE + 1000 * k)
    predictions = []

    for run in range(runs):
        idx = rng.choice(n, size=sample_n, replace=False)

        km = KMeans(
            n_clusters=k,
            n_init=N_INIT,
            random_state=RANDOM_STATE + 1000 * k + run,
        )
        km.fit(X[idx])

        # Predict all donors so every run is compared on the same population.
        predictions.append(km.predict(X))

    aris = []
    for i in range(len(predictions)):
        for j in range(i + 1, len(predictions)):
            aris.append(
                adjusted_rand_score(predictions[i], predictions[j])
            )

    aris = np.asarray(aris)

    return {
        "stability_ARI_mean": aris.mean(),
        "stability_ARI_p10": np.quantile(aris, 0.10),
    }


def k_diagnostics(Z, k_values=K_VALUES):
    rows = []
    X = Z.to_numpy()

    for k in k_values:
        if k >= len(X):
            continue

        km = KMeans(
            n_clusters=k,
            n_init=N_INIT,
            random_state=RANDOM_STATE,
        )
        labels = canonicalize_labels(km.fit_predict(X))

        counts = pd.Series(labels).value_counts(normalize=True)

        if len(X) > SILHOUETTE_SAMPLE_N:
            sil = silhouette_score(
                X,
                labels,
                sample_size=SILHOUETTE_SAMPLE_N,
                random_state=RANDOM_STATE,
            )
            sil_n = SILHOUETTE_SAMPLE_N
        else:
            sil = silhouette_score(X, labels)
            sil_n = len(X)

        stability = subsample_stability_ari(Z, k)

        rows.append({
            "K": k,
            "silhouette": sil,
            "stability_ARI_mean": stability["stability_ARI_mean"],
            "stability_ARI_p10": stability["stability_ARI_p10"],
            "smallest_cluster": counts.min(),
            "largest_cluster": counts.max(),
            "inertia_per_donor": km.inertia_ / len(X),
            "silhouette_n": sil_n,
        })

    return pd.DataFrame(rows)


K_DIAGNOSTICS = k_diagnostics(Z)

display(
    K_DIAGNOSTICS.style
    .format({
        "silhouette": "{:.3f}",
        "stability_ARI_mean": "{:.3f}",
        "stability_ARI_p10": "{:.3f}",
        "smallest_cluster": "{:.1%}",
        "largest_cluster": "{:.1%}",
        "inertia_per_donor": "{:.3f}",
        "silhouette_n": "{:,.0f}",
    })
    .hide(axis="index")
    .set_caption(
        "K diagnostics — silhouette = separation; "
        "stability ARI = reproducibility across 80% donor subsamples"
    )
)

K,silhouette,stability_ARI_mean,stability_ARI_p10,smallest_cluster,largest_cluster,inertia_per_donor,silhouette_n
3,0.172,0.979,0.966,31.9%,35.9%,3.312,"5,000"
4,0.183,0.973,0.959,17.8%,31.2%,2.964,"5,000"
5,0.173,0.978,0.969,16.4%,24.4%,2.698,"5,000"
6,0.180,0.976,0.963,8.5%,22.5%,2.540,"5,000"
7,0.177,0.968,0.952,7.7%,20.9%,2.394,"5,000"
8,0.172,0.962,0.943,5.5%,20.0%,2.286,"5,000"


## 6 — Fit the selected K

Cluster numbers are ordered largest-to-smallest for readability only.

In [7]:
def fit_selected(Z, k=SELECTED_K):
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    raw_labels = km.fit_predict(Z.to_numpy())
    labels = canonicalize_labels(raw_labels)
    assignments = pd.DataFrame({"cluster": labels}, index=Z.index)
    return km, assignments


MODEL, ASSIGNMENTS = fit_selected(Z, SELECTED_K)
DONORS_WITH_CLUSTER = DONORS.join(ASSIGNMENTS)
UNIT_SCORES_WITH_CLUSTER = Z_UNWEIGHTED.join(ASSIGNMENTS)

cluster_sizes = (
    ASSIGNMENTS["cluster"].value_counts().sort_index().rename("n").to_frame()
    .assign(share=lambda x: x["n"] / x["n"].sum())
)
display(
    cluster_sizes.style
    .format({"n": "{:,.0f}", "share": "{:.1%}"})
    .set_caption(f"Selected solution: K={SELECTED_K}")
)

,n,share
cluster,,
1,"12,265",22.5%
2,"11,731",21.5%
3,"9,492",17.4%
4,"8,513",15.6%
5,"7,994",14.6%
6,"4,636",8.5%


## 6A — Optionally score lower-history repeat donors

The cluster **fit remains unchanged**: only donors meeting the configured `MIN_GIFTS_FIELD >= MIN_GIFTS` threshold determine preprocessing, centroids, diagnostics, and cluster definitions.

When `SCORE_ALL_REPEAT = True`, donors with **2 through `MIN_GIFTS - 1`** on that same configured count field are transformed with the already-fitted medians/scalers and assigned to the nearest existing centroid. They are then appended to `DONORS_WITH_CLUSTER` and `UNIT_SCORES_WITH_CLUSTER`, so every downstream profiling/JTBD/executive-readout cell sees the broader repeat-donor population. `ASSIGNMENTS`, `DONORS`, `Z`, `MODEL`, and all fit diagnostics remain fit-population only.

The scoring output retains assignment distance/margin and raw-feature coverage because low-history donors have noisier behavioral estimates.


In [8]:
# ============================================================================
# OPTIONAL LOW-HISTORY REPEAT-DONOR SCORING
#
# IMPORTANT:
# - Does NOT refit preprocessing or K-means.
# - Does NOT change ASSIGNMENTS, DONORS, Z, MODEL, or K diagnostics.
# - If enabled, expands only the downstream profiling population.
# - Uses the SAME configured MIN_GIFTS_FIELD as the fit gate:
#       2 <= MIN_GIFTS_FIELD < MIN_GIFTS
# ============================================================================

_REQUIRED_SCORE_OBJECTS = [
    "FEATURES", "DONORS", "ASSIGNMENTS", "MODEL", "PREP", "Z", "Z_UNWEIGHTED", "UNITS_NOW"
]
_missing_score_objects = [name for name in _REQUIRED_SCORE_OBJECTS if name not in globals()]
if _missing_score_objects:
    raise NameError(
        "Run the notebook through the selected K fit before this scoring cell. "
        f"Missing: {_missing_score_objects}"
    )

# Rebuild these from fit-only artifacts every time so this cell is idempotent.
FIT_DONORS_WITH_CLUSTER = DONORS.join(ASSIGNMENTS).copy()
FIT_UNIT_SCORES_WITH_CLUSTER = Z_UNWEIGHTED.join(ASSIGNMENTS).copy()


def _transform_with_fitted_preprocessor(df, prep):
    """Apply the already-fitted training preprocessing to new donors."""
    fields = list(prep["fields"])
    units = prep["units"]

    missing_cols = [f for f in fields if f not in df.columns]
    if missing_cols:
        raise ValueError(
            "Scoring population is missing active fit fields: "
            + ", ".join(missing_cols)
        )

    X = df[fields].apply(pd.to_numeric, errors="coerce").copy()

    # Apply the identical project-cost coverage rule used during training.
    if PROJECT_COST_COVERAGE_FIELD in df.columns:
        cov = pd.to_numeric(df[PROJECT_COST_COVERAGE_FIELD], errors="coerce")
        low_cov = cov < PROJECT_COST_COVERAGE_FLOOR
        for f in PROJECT_COST_FIELDS.intersection(fields):
            X.loc[low_cov, f] = np.nan

    # Apply the identical transforms used during training.
    for f in fields:
        if f in LOG1P_FIELDS:
            if (X[f].dropna() < 0).any():
                raise ValueError(f"{f} has negative values but is declared log1p")
            X[f] = np.log1p(X[f])

    observed_before_impute = X.notna()
    X_imp = X.fillna(prep["medians"])

    if X_imp.isna().any().any():
        bad = X_imp.columns[X_imp.isna().any()].tolist()
        raise ValueError(f"Scoring still has missing active fields after fitted-median imputation: {bad}")

    feature_z = pd.DataFrame(
        prep["feature_scaler"].transform(X_imp[fields]),
        index=X_imp.index,
        columns=fields,
    )

    unit_raw = pd.DataFrame(index=X_imp.index)
    for unit, spec in units.items():
        signed = pd.DataFrame(
            {f: feature_z[f] * direction for f, direction in spec.items()},
            index=X_imp.index,
        )
        unit_raw[unit] = signed.mean(axis=1)

    unit_cols = list(units)
    unit_z = pd.DataFrame(
        prep["unit_scaler"].transform(unit_raw[unit_cols]),
        index=unit_raw.index,
        columns=unit_cols,
    )

    scoring_quality = pd.DataFrame(index=X_imp.index)
    scoring_quality["score_observed_fit_features"] = observed_before_impute.sum(axis=1).astype(int)
    scoring_quality["score_imputed_fit_features"] = (~observed_before_impute).sum(axis=1).astype(int)
    scoring_quality["score_fit_feature_coverage"] = observed_before_impute.mean(axis=1)

    return unit_z, scoring_quality


# Regression guard: applying the scoring transform back to fit donors must
# reproduce the already-built unweighted fit matrix exactly (within floating tolerance).
_check_n = min(64, len(DONORS))
_check_idx = DONORS.index[:_check_n]
_check_unit_z, _ = _transform_with_fitted_preprocessor(DONORS.loc[_check_idx], PREP)
if not np.allclose(
    _check_unit_z.to_numpy(),
    Z_UNWEIGHTED.loc[_check_idx, _check_unit_z.columns].to_numpy(),
    rtol=1e-10,
    atol=1e-10,
):
    raise RuntimeError(
        "Scoring preprocessing does not reproduce the fitted unit scores. "
        "Stop before assigning lower-history donors."
    )


def _raw_to_canonical_cluster_map(model, Z_fit, canonical_assignments):
    """Recover the exact raw KMeans-label -> displayed canonical-label mapping."""
    raw_fit = model.predict(Z_fit.to_numpy())
    canonical_fit = canonical_assignments.loc[Z_fit.index, "cluster"].to_numpy()

    mapping = {}
    for raw_label in np.unique(raw_fit):
        mapped = np.unique(canonical_fit[raw_fit == raw_label])
        if len(mapped) != 1:
            raise RuntimeError(
                f"Could not recover a one-to-one canonical label for raw KMeans label {raw_label}: {mapped}"
            )
        mapping[int(raw_label)] = int(mapped[0])

    if len(mapping) != SELECTED_K:
        raise RuntimeError(
            f"Recovered {len(mapping)} KMeans labels but SELECTED_K={SELECTED_K}."
        )

    return mapping


# Default/disabled state is exactly the original fit-only profiling population.
SCORED_REPEAT_DONORS = FEATURES.iloc[0:0].copy().set_index("donor_id")
SCORED_REPEAT_ASSIGNMENTS = pd.DataFrame(
    columns=[
        "cluster", "assignment_source", "score_count_field", "score_count",
        "score_observed_fit_features", "score_imputed_fit_features",
        "score_fit_feature_coverage", "assignment_distance",
        "second_distance", "assignment_margin", "assignment_margin_ratio",
    ],
    index=pd.Index([], name="donor_id"),
)
SCORED_REPEAT_UNIT_SCORES = pd.DataFrame(
    columns=list(Z_UNWEIGHTED.columns),
    index=pd.Index([], name="donor_id"),
    dtype=float,
)

DONORS_WITH_CLUSTER = FIT_DONORS_WITH_CLUSTER.copy()
UNIT_SCORES_WITH_CLUSTER = FIT_UNIT_SCORES_WITH_CLUSTER.copy()
PROFILE_ASSIGNMENTS = ASSIGNMENTS.copy()

if SCORE_ALL_REPEAT:
    score_count = pd.to_numeric(FEATURES[MIN_GIFTS_FIELD], errors="coerce")
    score_mask = (
        pd.to_numeric(FEATURES["cluster_eligible"], errors="coerce").eq(1)
        & score_count.ge(2)
        & score_count.lt(MIN_GIFTS)
    )

    if EXCLUDE_TEACHERS:
        if "is_teacher" not in FEATURES.columns:
            raise ValueError("EXCLUDE_TEACHERS=True but is_teacher is not available for scoring.")
        score_mask &= ~pd.to_numeric(FEATURES["is_teacher"], errors="coerce").eq(1)

    SCORED_REPEAT_DONORS = FEATURES.loc[score_mask].copy().set_index("donor_id")

    overlap = SCORED_REPEAT_DONORS.index.intersection(DONORS.index)
    if len(overlap):
        raise RuntimeError(
            f"Scoring population overlaps the fit population for {len(overlap):,} donors. "
            "The population gates should be mutually exclusive."
        )

    if len(SCORED_REPEAT_DONORS):
        SCORED_REPEAT_UNIT_SCORES, _score_quality = _transform_with_fitted_preprocessor(
            SCORED_REPEAT_DONORS,
            PREP,
        )

        # Apply the same unit weights used by the fitted K-means geometry.
        _score_Z_weighted = SCORED_REPEAT_UNIT_SCORES.copy()
        for unit in _score_Z_weighted.columns:
            weight = float(UNIT_WEIGHTS.get(unit, 1.0))
            if weight <= 0:
                raise ValueError(f"Unit weight must be > 0: {unit}={weight}")
            _score_Z_weighted[unit] = _score_Z_weighted[unit] * np.sqrt(weight)

        # Exact column order used to fit MODEL.
        _score_Z_weighted = _score_Z_weighted.loc[:, list(Z.columns)]

        _label_map = _raw_to_canonical_cluster_map(MODEL, Z, ASSIGNMENTS)

        # Regression guard: raw-model labels mapped back to canonical labels
        # must exactly reproduce the selected fit assignments.
        _fit_raw_pred = MODEL.predict(Z.to_numpy())
        _fit_canonical_pred = np.array([_label_map[int(x)] for x in _fit_raw_pred], dtype=int)
        if not np.array_equal(_fit_canonical_pred, ASSIGNMENTS.loc[Z.index, "cluster"].to_numpy()):
            raise RuntimeError(
                "Canonical label recovery does not reproduce the selected fit assignments."
            )

        _raw_pred = MODEL.predict(_score_Z_weighted.to_numpy())
        _canonical_pred = np.array([_label_map[int(x)] for x in _raw_pred], dtype=int)

        # Assignment diagnostics: nearest vs second-nearest centroid distance.
        _distances = MODEL.transform(_score_Z_weighted.to_numpy())
        _row_idx = np.arange(len(_distances))
        _nearest = _distances[_row_idx, _raw_pred]
        _second = np.partition(_distances, kth=1, axis=1)[:, 1]
        _margin = _second - _nearest
        _margin_ratio = np.divide(
            _margin,
            _second,
            out=np.full(len(_margin), np.nan, dtype=float),
            where=_second > 0,
        )

        SCORED_REPEAT_ASSIGNMENTS = pd.DataFrame(
            {
                "cluster": _canonical_pred,
                "assignment_source": "scored_repeat",
                "score_count_field": MIN_GIFTS_FIELD,
                "score_count": score_count.loc[score_mask].to_numpy(),
                "score_observed_fit_features": _score_quality["score_observed_fit_features"].to_numpy(),
                "score_imputed_fit_features": _score_quality["score_imputed_fit_features"].to_numpy(),
                "score_fit_feature_coverage": _score_quality["score_fit_feature_coverage"].to_numpy(),
                "assignment_distance": _nearest,
                "second_distance": _second,
                "assignment_margin": _margin,
                "assignment_margin_ratio": _margin_ratio,
            },
            index=SCORED_REPEAT_DONORS.index,
        )

        # Add provenance to the fit rows only when the expanded population is enabled.
        _fit_assignments_extended = ASSIGNMENTS.copy()
        _fit_assignments_extended["assignment_source"] = "fit"
        _fit_assignments_extended["score_count_field"] = MIN_GIFTS_FIELD
        _fit_assignments_extended["score_count"] = pd.to_numeric(
            DONORS[MIN_GIFTS_FIELD], errors="coerce"
        )
        for c in [
            "score_observed_fit_features", "score_imputed_fit_features",
            "score_fit_feature_coverage", "assignment_distance",
            "second_distance", "assignment_margin", "assignment_margin_ratio",
        ]:
            _fit_assignments_extended[c] = np.nan

        _fit_profile = DONORS.join(_fit_assignments_extended)
        _scored_profile = SCORED_REPEAT_DONORS.join(SCORED_REPEAT_ASSIGNMENTS)

        DONORS_WITH_CLUSTER = pd.concat([_fit_profile, _scored_profile], axis=0).sort_index()
        UNIT_SCORES_WITH_CLUSTER = pd.concat(
            [
                Z_UNWEIGHTED.join(ASSIGNMENTS),
                SCORED_REPEAT_UNIT_SCORES.join(SCORED_REPEAT_ASSIGNMENTS[["cluster"]]),
            ],
            axis=0,
        ).sort_index()

        PROFILE_ASSIGNMENTS = pd.concat(
            [_fit_assignments_extended, SCORED_REPEAT_ASSIGNMENTS],
            axis=0,
        ).sort_index()

# Strong invariants: scoring must never mutate the fit artifacts.
if len(ASSIGNMENTS) != len(DONORS) or not ASSIGNMENTS.index.equals(DONORS.index):
    raise RuntimeError("Fit assignments were unexpectedly altered while scoring repeat donors.")
if not Z.index.equals(DONORS.index):
    raise RuntimeError("Fit matrix Z was unexpectedly altered while scoring repeat donors.")
if not UNIT_SCORES_WITH_CLUSTER.index.equals(DONORS_WITH_CLUSTER.index):
    raise RuntimeError("Profile donor rows and profile unit-score rows are misaligned after scoring.")
if DONORS_WITH_CLUSTER.index.duplicated().any():
    raise RuntimeError("Duplicate donor_id found in the expanded profiling population.")

PROFILE_CLUSTER_SIZES = (
    DONORS_WITH_CLUSTER["cluster"].value_counts().sort_index().rename("profile_n").to_frame()
    .assign(profile_share=lambda t: t["profile_n"] / t["profile_n"].sum())
)
_fit_n_by_cluster = ASSIGNMENTS["cluster"].value_counts().sort_index().rename("fit_n")
_score_n_by_cluster = (
    SCORED_REPEAT_ASSIGNMENTS["cluster"].value_counts().sort_index().rename("scored_repeat_n")
    if len(SCORED_REPEAT_ASSIGNMENTS) else pd.Series(dtype=int, name="scored_repeat_n")
)
PROFILE_CLUSTER_SIZES = (
    PROFILE_CLUSTER_SIZES
    .join(_fit_n_by_cluster, how="left")
    .join(_score_n_by_cluster, how="left")
    .fillna({"fit_n": 0, "scored_repeat_n": 0})
)
PROFILE_CLUSTER_SIZES[["fit_n", "scored_repeat_n", "profile_n"]] = (
    PROFILE_CLUSTER_SIZES[["fit_n", "scored_repeat_n", "profile_n"]].astype(int)
)

if SCORE_ALL_REPEAT:
    _score_range_label = f"2 <= {MIN_GIFTS_FIELD} < {MIN_GIFTS}"
    _median_margin = (
        SCORED_REPEAT_ASSIGNMENTS["assignment_margin_ratio"].median()
        if len(SCORED_REPEAT_ASSIGNMENTS) else np.nan
    )
    display(HTML(f"""
    <div style='border:1px solid #d1d5db;border-radius:8px;padding:12px 14px;margin:8px 0 12px;background:#f8fafc'>
      <div style='font-size:16px;font-weight:700'>Repeat-donor scoring enabled</div>
      <div style='margin-top:4px'>
        Fit remains <b>{len(DONORS):,}</b> donors with <code>{MIN_GIFTS_FIELD} &gt;= {MIN_GIFTS}</code>.
        Added <b>{len(SCORED_REPEAT_DONORS):,}</b> lower-history repeat donors satisfying
        <code>{escape(_score_range_label)}</code> to downstream profiling only.
      </div>
      <div style='margin-top:4px;color:#555'>
        Expanded profiling population: <b>{len(DONORS_WITH_CLUSTER):,}</b> donors.
        Median scored-donor assignment margin ratio: <b>{_median_margin:.1%}</b>.
      </div>
    </div>
    """))
else:
    display(HTML("""
    <div style='border:1px solid #d1d5db;border-radius:8px;padding:10px 12px;margin:8px 0 12px;background:#f8fafc'>
      <b>Repeat-donor scoring disabled.</b> Downstream profiling remains fit-population only.
    </div>
    """))

_display_profile_sizes = PROFILE_CLUSTER_SIZES.copy()
display(
    _display_profile_sizes.style
    .format({
        "fit_n": "{:,.0f}",
        "scored_repeat_n": "{:,.0f}",
        "profile_n": "{:,.0f}",
        "profile_share": "{:.1%}",
    })
    .set_caption("Population flowing into downstream cluster profiling")
)

print(
    "Scoring objects ready: SCORED_REPEAT_DONORS, SCORED_REPEAT_ASSIGNMENTS, "
    "SCORED_REPEAT_UNIT_SCORES, PROFILE_ASSIGNMENTS, PROFILE_CLUSTER_SIZES"
)


,profile_n,profile_share,fit_n,scored_repeat_n
cluster,,,,
1,"12,265",22.5%,"12,265",0
2,"11,731",21.5%,"11,731",0
3,"9,492",17.4%,"9,492",0
4,"8,513",15.6%,"8,513",0
5,"7,994",14.6%,"7,994",0
6,"4,636",8.5%,"4,636",0


Scoring objects ready: SCORED_REPEAT_DONORS, SCORED_REPEAT_ASSIGNMENTS, SCORED_REPEAT_UNIT_SCORES, PROFILE_ASSIGNMENTS, PROFILE_CLUSTER_SIZES


## 7 — Clean cluster profiles

Each cluster card has three levels:

1. **Behavioral dimensions** — the unit scores K-means actually saw.
2. **Raw components** — the observable behaviors behind those scores.
3. **Strongest profile-only differences** — context that did not influence the fit.

Distinctiveness is descriptive, not causal importance.

In [9]:
def is_rate_like(field, series):
    name = field.lower()
    observed = pd.to_numeric(series, errors="coerce").dropna()
    bounded = len(observed) and observed.min() >= -1e-9 and observed.max() <= 1.000001
    return name.startswith(("share_", "is_", "has_")) or "_rate" in name or bounded


def raw_summary(field, series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if not len(x):
        return np.nan, "mean"
    if is_rate_like(field, x):
        return float(x.mean()), "mean"
    return float(x.median()), "median"


def format_raw(field, value, method, reference_series):
    if pd.isna(value):
        return "-"
    name = field.lower()
    if is_rate_like(field, reference_series):
        return f"{value:.0%}"
    if "amount" in name and "share_" not in name and "rate" not in name and "ratio" not in name:
        return f"${value:,.0f}"
    if "ratio" in name:
        return f"{value:.2f}x"
    if any(token in name for token in ["days", "months", "count", "n_unique", "n_gifts", "visits", "events"]):
        return f"{value:,.1f}"
    return f"{value:,.2f}"


def build_feature_profile_long(df_with_cluster, units, profile_fields=PROFILE_FIELDS):
    fit_fields = flatten_unit_features(units)
    field_to_unit = {f: unit for unit, spec in units.items() for f in spec}
    available_profile = [f for f in profile_fields if f in df_with_cluster.columns and f not in fit_fields]
    fields = list(dict.fromkeys(fit_fields + available_profile))

    numeric = df_with_cluster[fields].apply(pd.to_numeric, errors="coerce")
    overall_mean = numeric.mean()
    overall_std = numeric.std(ddof=0).replace(0, np.nan)
    coverage = numeric.notna().mean()

    overall_raw = {}
    methods = {}
    for f in fields:
        overall_raw[f], methods[f] = raw_summary(f, numeric[f])

    rows = []
    for cluster, idx in df_with_cluster.groupby("cluster").groups.items():
        sub = numeric.loc[idx]
        effect = (sub.mean() - overall_mean) / overall_std

        for f in fields:
            cv, _ = raw_summary(f, sub[f])
            rows.append({
                "cluster": int(cluster),
                "feature": f,
                "source": "COMPONENT" if f in fit_fields else "PROFILE",
                "unit": field_to_unit.get(f, "profile-only"),
                "cluster_value": cv,
                "overall_value": overall_raw[f],
                "raw_method": methods[f],
                "effect_z": float(effect[f]) if pd.notna(effect[f]) else np.nan,
                "abs_effect_z": float(abs(effect[f])) if pd.notna(effect[f]) else np.nan,
                "coverage": float(coverage[f]),
            })

    return pd.DataFrame(rows)


FEATURE_PROFILE_LONG = build_feature_profile_long(DONORS_WITH_CLUSTER, UNITS_NOW)


def build_unit_profile_long(unit_scores_with_cluster):
    rows = []
    unit_cols = [c for c in unit_scores_with_cluster.columns if c != "cluster"]

    for cluster, sub in unit_scores_with_cluster.groupby("cluster"):
        for unit in unit_cols:
            value = float(sub[unit].mean())
            rows.append({
                "cluster": int(cluster),
                "unit": unit,
                "score": value,
                "abs_score": abs(value),
            })

    return pd.DataFrame(rows)


UNIT_PROFILE_LONG = build_unit_profile_long(UNIT_SCORES_WITH_CLUSTER)


def effect_bar(z):
    if pd.isna(z):
        return '<span style="color:#777">-</span>'

    width = min(abs(float(z)) / 2.0, 1.0) * 100
    fill = "#dbeafe" if z > 0 else "#fee2e2"
    label = f"{z:+.2f} SD"

    return (
        '<div style="min-width:145px">'
        '<div style="height:18px;background:#f3f4f6;border-radius:3px;position:relative;overflow:hidden">'
        f'<div style="height:100%;width:{width:.1f}%;background:{fill}"></div>'
        f'<span style="position:absolute;left:6px;top:1px;font-size:12px;font-weight:600">{label}</span>'
        '</div></div>'
    )


def unit_table_html(rows):
    body = []
    for r in rows.itertuples(index=False):
        body.append(
            "<tr>"
            f"<td style='font-size:13px;font-weight:600'>{escape(str(r.unit).replace('_', ' '))}</td>"
            f"<td style='text-align:right;font-weight:600'>{r.score:+.2f}</td>"
            f"<td>{effect_bar(r.score)}</td>"
            "</tr>"
        )

    return (
        "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
        "<thead><tr style='background:#f8fafc'>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Behavioral dimension</th>"
        "<th style='text-align:right;padding:6px;border-bottom:1px solid #ddd'>Mean score</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Distinctiveness</th>"
        "</tr></thead><tbody>"
        + "".join(body)
        + "</tbody></table>"
    )


def feature_table_html(rows, df_reference):
    body = []

    for r in rows.itertuples(index=False):
        cv = format_raw(r.feature, r.cluster_value, r.raw_method, df_reference[r.feature])
        ov = format_raw(r.feature, r.overall_value, r.raw_method, df_reference[r.feature])

        body.append(
            "<tr>"
            f"<td style='font-family:ui-monospace,monospace;font-size:12px'>{escape(r.feature)}</td>"
            f"<td style='font-size:12px;color:#555'>{escape(str(r.unit))}</td>"
            f"<td style='text-align:right;font-weight:600'>{cv}</td>"
            f"<td style='text-align:right;color:#666'>{ov}</td>"
            f"<td>{effect_bar(r.effect_z)}</td>"
            "</tr>"
        )

    return (
        "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
        "<thead><tr style='background:#f8fafc'>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Feature</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Unit</th>"
        "<th style='text-align:right;padding:6px;border-bottom:1px solid #ddd'>Cluster</th>"
        "<th style='text-align:right;padding:6px;border-bottom:1px solid #ddd'>Overall</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Distinctiveness</th>"
        "</tr></thead><tbody>"
        + "".join(body)
        + "</tbody></table>"
    )


def show_cluster_profiles(
    unit_profile=UNIT_PROFILE_LONG,
    feature_profile=FEATURE_PROFILE_LONG,
    df_reference=DONORS_WITH_CLUSTER,
    top_profile_n=PROFILE_TOP_N,
):
    total_n = len(df_reference)

    for cluster in sorted(feature_profile["cluster"].unique()):
        n = int((df_reference["cluster"] == cluster).sum())

        unit_rows = unit_profile.loc[
            unit_profile["cluster"] == cluster
        ].sort_values("abs_score", ascending=False)

        component_rows = feature_profile.loc[
            (feature_profile["cluster"] == cluster)
            & (feature_profile["source"] == "COMPONENT")
        ].sort_values(["unit", "abs_effect_z"], ascending=[True, False])

        profile_rows = feature_profile.loc[
            (feature_profile["cluster"] == cluster)
            & (feature_profile["source"] == "PROFILE")
            & (feature_profile["coverage"] >= PROFILE_MIN_COVERAGE)
        ].sort_values("abs_effect_z", ascending=False).head(top_profile_n)

        html = f"""
        <div style='border:1px solid #d1d5db;border-radius:8px;padding:14px 16px;margin:14px 0 22px 0;background:white'>
          <div style='font-size:21px;font-weight:700;margin-bottom:2px'>Cluster {cluster}</div>
          <div style='color:#555;margin-bottom:14px'>{n:,} donors - {n/total_n:.1%} of clustered population</div>

          <div style='font-size:15px;font-weight:700;margin:8px 0'>Behavioral dimensions used by K-means</div>
          {unit_table_html(unit_rows)}

          <div style='font-size:15px;font-weight:700;margin:16px 0 8px 0'>Raw components behind those dimensions</div>
          {feature_table_html(component_rows, df_reference)}

          <div style='font-size:15px;font-weight:700;margin:16px 0 8px 0'>Strongest profile-only differences</div>
          {feature_table_html(profile_rows, df_reference)}
        </div>
        """

        display(HTML(html))


show_cluster_profiles()

## 8 — Optional experiments

The baseline stays untouched. Add one or more retained optional units and compare assignments with the baseline using ARI.

**ARI measures assignment agreement, not model quality.**

In [10]:
def run_experiment(optional_units, k=SELECTED_K):
    units = active_units(optional_units)
    required = flatten_unit_features(units)
    missing = [f for f in required if f not in DONORS.columns]

    if missing:
        raise ValueError(f"Experiment requires fields that were not available/loaded: {missing}")

    Z_alt, prep_alt = prepare_unit_matrix(DONORS, units)
    km_alt = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    labels_alt = canonicalize_labels(km_alt.fit_predict(Z_alt.to_numpy()))
    assignments_alt = pd.DataFrame({"cluster": labels_alt}, index=Z_alt.index)

    return {
        "optional_units": list(optional_units),
        "units": units,
        "Z": Z_alt,
        "prep": prep_alt,
        "model": km_alt,
        "assignments": assignments_alt,
    }


def compare_to_core(optional_units, k=SELECTED_K):
    alt = run_experiment(optional_units, k=k)
    common = ASSIGNMENTS.index.intersection(alt["assignments"].index)

    ari = adjusted_rand_score(
        ASSIGNMENTS.loc[common, "cluster"],
        alt["assignments"].loc[common, "cluster"],
    )

    return pd.DataFrame([{
        "experiment": "core + " + " + ".join(optional_units),
        "K": k,
        "units": len(alt["units"]),
        "ARI_vs_core": ari,
        "same_donors_n": len(common),
    }])


# Examples:
# display(compare_to_core(["campaign_responsiveness"]))
# display(compare_to_core(["jtbd_values_equity"]))
# display(compare_to_core(["site_engagement"]))

print("Optional experiment helpers are ready.")
print("Available optional units:")
print(sorted(OPTIONAL_UNIT_LIBRARY))

Optional experiment helpers are ready.
Available optional units:
['campaign_responsiveness', 'choice_breadth', 'funding_depth', 'jtbd_active_participation', 'jtbd_decision_deliberation', 'jtbd_giving_leverage', 'jtbd_local_stewardship', 'jtbd_recognition_avoidance', 'jtbd_sustained_monthly', 'jtbd_tax_efficiency', 'jtbd_values_equity', 'platform_support', 'seasonality', 'site_engagement', 'trigger']


## 9 — Optional export

Useful objects:
- `ASSIGNMENTS` — fit-population donor → cluster (unchanged by optional scoring)
- `PROFILE_ASSIGNMENTS` — fit + optionally scored lower-history repeat donors → cluster
- `Z` — standardized unit scores K-means actually used
- `UNIT_PROFILE_LONG` — cluster profiles on behavioral dimensions
- `FEATURE_PROFILE_LONG` — raw components + profile-only fields
- `K_DIAGNOSTICS` — K diagnostics

In [11]:
# EXPORT_DIR = DATA_DIR.parent / "Clustering Runs - EFA Informed"
# EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# ASSIGNMENTS.reset_index().to_csv(EXPORT_DIR / "cluster_assignments.csv", index=False)
# Z.reset_index().to_csv(EXPORT_DIR / "unit_scores.csv", index=False)
# UNIT_PROFILE_LONG.to_csv(EXPORT_DIR / "unit_profiles.csv", index=False)
# FEATURE_PROFILE_LONG.to_csv(EXPORT_DIR / "feature_profiles.csv", index=False)
# K_DIAGNOSTICS.to_csv(EXPORT_DIR / "k_diagnostics.csv", index=False)
# print(f"Wrote outputs to {EXPORT_DIR}")

print("Nothing exported by default.")

Nothing exported by default.


In [12]:
# ============================================================================
# Behavioral JTBD profiling — donor-level ranking + cluster summaries
#
# Interpretation:
# These are behavioral signals CONSISTENT WITH a Job, not proof that the donor
# consciously holds that motivation.
#
# IMPORTANT:
# - Top-N comparisons use COMPLETE CASES across all rankable Jobs.
# - Evidence-only Jobs are scored but cannot consume Top-N slots.
# - A Job is called a "standout" only when both relative AND absolute evidence
#   clear the thresholds below.
# ============================================================================

import numpy as np
import pandas as pd

from IPython.display import HTML, Markdown, display


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------

JTBD_TOP_N = 2

# Otherwise-rankable Job must be measurable for at least this share of donors.
JTBD_MIN_JOB_COVERAGE = 0.55

# Business-facing standout rules.
# These are display / interpretation rules, not statistical significance tests.
JTBD_STANDOUT_MIN_INDEX = 1.25
JTBD_STANDOUT_MIN_EVIDENCE_Z = 0.20

# Maximum standout Jobs shown on each cluster card.
JTBD_SHOW_TOP_PER_CLUSTER = 5


# ----------------------------------------------------------------------------
# JOB DEFINITIONS
#
# Feature value = signed within-Job weight.
#
# Positive = more feature -> more evidence for Job
# Negative = less feature -> more evidence for Job
#
# Each completed Job score is standardized across donors before Jobs are
# compared within a donor.
# ----------------------------------------------------------------------------

JTBD_JOB_SPECS = {

    # ------------------------------------------------------------------------
    # FUNCTIONAL
    # ------------------------------------------------------------------------

    "Concrete impact": {
        "definition": "Produce a tangible, understandable result",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Keep only one direct funding-depth measure so this does not
            # simply become "large giver."
            "median_gift_to_project_cost_ratio_24m": +0.75,

            # Different manifestation: finishing a concrete project need.
            "share_gifts_closed_project_24m": +1.00,

            # Tangible item/list giving; useful but product-opportunity dependent.
            "share_gifts_classroom_essentials_24m": +0.50,
        },
    },

    "Directed choice": {
        "definition": "Understand the specific project or destination my gift will support",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_with_prior_search_24m": +1.00,
            "project_page_pre_gift_mean_24m": +1.00,
        },
    },

    "Giving leverage": {
        "definition": "Make my contribution accomplish more by unlocking additional funds",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_with_match_24m": +1.00,
            "mean_match_excess_24m": +1.00,
        },
    },

    "Tax efficiency": {
        "definition": "Leverage charitable tax deductions for my financial benefit",

        # Useful directional evidence, but not strong enough to compete
        # against better-observed Jobs for a donor's Top-2 slots.
        "rankable": False,
        "min_components": 2,
        "features": {
            "share_gifts_daf_24m": +2.00,
            "share_gifts_year_end_final_week_24m": +1.00,
            "share_gifts_q4_24m": +0.50,
        },
    },

    "Sustained giving": {
        "definition": "Make generosity a dependable practice",
        "rankable": True,
        "min_components": 3,
        "features": {
            # Recent consistency.
            "n_active_months_24m": +1.00,
            "entropy_gift_month_norm_24m": +1.00,

            # Persistence across windows.
            "is_continuing_donor_24m": +1.00,

            # Length of overall giving relationship.
            "tenure_days_any_giving": +0.75,

            # Only one explicitly monthly-specific measure.
            "monthly_longest_streak_months": +0.50,
        },
    },

    "Confidence and risk reduction": {
        "definition": "Avoid an unsafe, ineffective, or regrettable giving decision",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Repeated school support as an observed trust/persistence signal.
            "share_gifts_repeat_school_24m": +1.00,

            # Research / deliberation.
            "share_gifts_with_prior_search_24m": +0.75,
            "teacher_page_pre_gift_mean_24m": +0.50,
        },
    },

    # ------------------------------------------------------------------------
    # SOCIAL
    # ------------------------------------------------------------------------

    "Relational support": {
        "definition": "Show up for someone I care about",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_repeat_teacher_24m": +1.00,
            "top_teacher_share_count_24m": +1.00,
            "gifts_per_teacher_24m": +0.75,

            # Relationship-origin context, weaker than observed repeated support.
            "is_teacher_referred": +0.50,
        },
    },

    "Local stewardship": {
        "definition": "Strengthen the community where I live",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_within_15mi_24m": +1.00,
            "median_distance_mi_24m": -1.00,
            "share_gifts_same_state_24m": +0.75,
        },
    },

    "Collective participation": {
        "definition": "Join others I trust in supporting something together",

        # We can see some relevant behavior, but sharing/referral evidence is
        # currently too sparse/contextual to compete for Top-2 slots.
        "rankable": False,
        "min_components": 2,
        "features": {
            "sharing_active_months_24m": +1.00,
            "share_gifts_channel_sharetray_24m": +1.00,
            "share_gifts_channel_facebook_24m": +0.50,
            "share_gifts_channel_nextdoor_24m": +0.50,
        },
    },

    # ------------------------------------------------------------------------
    # EMOTIONAL / CROSS-CUTTING
    # ------------------------------------------------------------------------

    "Urgent action": {
        "definition": "Act when a need becomes urgent and emotionally real",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Direct project urgency.
            "days_gift_to_expiration_median_24m": -1.00,
            "share_gifts_expiring_soon_24m": +1.00,

            # Supporting late-cycle evidence.
            "share_gifts_late_cycle_24m": +0.50,

            # Separate manifestation: speed of action after encountering need.
            "share_gifts_same_day_first_session_24m": +0.50,
            "days_first_session_to_gift_median_24m": -0.50,
        },
    },

    "Values obligation": {
        "definition": "Feel true to my personal values by giving",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Specifically equity-oriented behavioral expressions of Values.
            "share_gifts_to_low_income_schools_24m": +1.00,
            "share_gifts_to_historically_underrepresented_race_schools_24m": +1.00,
            "share_gifts_to_underserved_rural_schools_24m": +1.00,
        },
    },
}


# Jobs where current 1P data cannot support a defensible behavioral score.
JTBD_NOT_RANKED = {

    "Impact witness":
        "No clean post-gift sequence showing that the donor actually follows "
        "the work or consumes evidence of progress.",

    "Active participation":
        "Current first-party data do not directly observe taking part in the "
        "work beyond giving money. Sharing is treated as Collective "
        "Participation instead.",

    "Acknowledged contribution":
        "Anonymous giving is a narrow inverse proxy, but is not enough to "
        "support a multi-signal behavioral score for desire for recognition.",

    "Faith obligation":
        "No defensible behavioral signal in the current first-party feature build.",
}


# ----------------------------------------------------------------------------
# 1. BUILD PROFILING FRAME
# ----------------------------------------------------------------------------

if "DONORS_WITH_CLUSTER" not in globals():
    raise NameError(
        "Run the clustering notebook through DONORS_WITH_CLUSTER before this cell."
    )

JTBD_BASE = DONORS_WITH_CLUSTER.copy()

if JTBD_BASE.index.name != "donor_id" and "donor_id" in JTBD_BASE.columns:
    JTBD_BASE = JTBD_BASE.set_index("donor_id")


_all_jtbd_fields = sorted({
    f
    for spec in JTBD_JOB_SPECS.values()
    for f in spec["features"]
})

_control_fields = [
    "coverage_project_cost_24m",
]

_missing = [
    f
    for f in _all_jtbd_fields + _control_fields
    if f not in JTBD_BASE.columns
]


# Pull missing JTBD fields from original feature CSV when possible.
if (
    _missing
    and "FEATURES_PATH" in globals()
    and FEATURES_PATH.exists()
):

    _header = set(
        pd.read_csv(
            FEATURES_PATH,
            nrows=0,
        ).columns
    )

    _loadable = [
        f for f in _missing
        if f in _header
    ]

    if _loadable:
        _extra = (
            pd.read_csv(
                FEATURES_PATH,
                usecols=["donor_id"] + _loadable,
            )
            .set_index("donor_id")
        )

        JTBD_BASE = JTBD_BASE.join(
            _extra,
            how="left",
        )


# Same project-cost coverage guardrail used in clustering.
_project_cost_jtbd_fields = {
    "median_gift_to_project_cost_ratio_24m",
    "share_gifts_over_half_project_cost_24m",
    "share_gifts_full_project_cost_24m",
}

if "coverage_project_cost_24m" in JTBD_BASE.columns:

    _low_cov = (
        pd.to_numeric(
            JTBD_BASE["coverage_project_cost_24m"],
            errors="coerce",
        )
        .lt(0.50)
    )

    for _f in _project_cost_jtbd_fields.intersection(
        JTBD_BASE.columns
    ):
        JTBD_BASE.loc[_low_cov, _f] = np.nan


# ----------------------------------------------------------------------------
# 2. SCORE EACH JOB
#
# 1. Convert raw features to donor percentile ranks.
# 2. Orient signs consistently.
# 3. Weighted-average components within each Job.
# 4. Standardize the final Job score across donors.
# ----------------------------------------------------------------------------

JTBD_JOB_SCORES_RAW = pd.DataFrame(
    index=JTBD_BASE.index
)

_audit_rows = []


for _job, _spec in JTBD_JOB_SPECS.items():

    _available = [
        f
        for f in _spec["features"]
        if f in JTBD_BASE.columns
    ]

    _component_frame = pd.DataFrame(
        index=JTBD_BASE.index
    )

    _weights = {}


    for _f in _available:

        _x = pd.to_numeric(
            JTBD_BASE[_f],
            errors="coerce",
        )

        # Ignore unusable / constant fields.
        if (
            _x.notna().sum() < 2
            or _x.dropna().nunique() <= 1
        ):
            continue


        _signed_weight = float(
            _spec["features"][_f]
        )

        _weight = abs(
            _signed_weight
        )


        if _weight <= 0:
            continue


        _pct = _x.rank(
            method="average",
            pct=True,
        )


        if _signed_weight < 0:
            _pct = 1.0 - _pct


        _component_frame[_f] = _pct
        _weights[_f] = _weight


    _usable_components = list(
        _component_frame.columns
    )

    _min_components = int(
        _spec["min_components"]
    )


    if len(_usable_components) >= _min_components:

        _n_observed = (
            _component_frame
            .notna()
            .sum(axis=1)
        )


        _weighted_values = pd.DataFrame(
            {
                f: _component_frame[f] * _weights[f]
                for f in _usable_components
            },
            index=JTBD_BASE.index,
        )

        _observed_weights = pd.DataFrame(
            {
                f: _component_frame[f].notna().astype(float) * _weights[f]
                for f in _usable_components
            },
            index=JTBD_BASE.index,
        )


        _score = (
            _weighted_values
            .sum(
                axis=1,
                skipna=True,
            )
            /
            _observed_weights
            .sum(axis=1)
            .replace(0, np.nan)
        )


        _score = _score.where(
            _n_observed >= _min_components
        )

    else:

        _score = pd.Series(
            np.nan,
            index=JTBD_BASE.index,
        )


    JTBD_JOB_SCORES_RAW[_job] = _score


    _used_text = " | ".join(
        f"{f} ({_spec['features'][f]:+.2f})"
        for f in _usable_components
    )


    _audit_rows.append({
        "Job": _job,
        "intended status":
            "RANKABLE"
            if _spec.get("rankable", True)
            else "EVIDENCE ONLY",
        "defined features": len(
            _spec["features"]
        ),
        "usable features": len(
            _usable_components
        ),
        "features used":
            _used_text
            if _used_text
            else "—",
        "donor coverage":
            float(_score.notna().mean()),
        "definition":
            _spec["definition"],
    })


JTBD_JOB_AUDIT = pd.DataFrame(
    _audit_rows
)


# Standardize completed Job scores across donors.
JTBD_JOB_SCORES_Z = (
    JTBD_JOB_SCORES_RAW.copy()
)

for _job in JTBD_JOB_SCORES_Z.columns:

    _s = JTBD_JOB_SCORES_Z[_job]

    _sd = _s.std(
        ddof=0
    )

    if (
        pd.notna(_sd)
        and _sd > 0
    ):

        JTBD_JOB_SCORES_Z[_job] = (
            (_s - _s.mean()) / _sd
        )

    else:

        JTBD_JOB_SCORES_Z[_job] = np.nan


# ----------------------------------------------------------------------------
# 3. DETERMINE RANKABLE VS EVIDENCE-ONLY JOBS
# ----------------------------------------------------------------------------

_rankable_jobs = [

    _job

    for _job, _spec in JTBD_JOB_SPECS.items()

    if _spec.get(
        "rankable",
        True,
    )

    and (
        JTBD_JOB_SCORES_Z[_job]
        .notna()
        .mean()
        >= JTBD_MIN_JOB_COVERAGE
    )

    and (
        JTBD_JOB_SCORES_Z[_job]
        .notna()
        .sum()
        > 1
    )
]


if len(_rankable_jobs) < JTBD_TOP_N:

    raise ValueError(
        f"Only {len(_rankable_jobs)} Jobs meet the ranking rules; "
        f"cannot assign Top {JTBD_TOP_N}. "
        "Inspect JTBD_JOB_AUDIT."
    )


def _final_status(row):

    if row["intended status"] == "EVIDENCE ONLY":
        return "EVIDENCE ONLY"

    if row["Job"] in _rankable_jobs:
        return "RANKED"

    return "EVIDENCE ONLY / LOW COVERAGE"


JTBD_JOB_AUDIT[
    "Top-N status"
] = JTBD_JOB_AUDIT.apply(
    _final_status,
    axis=1,
)


_evidence_only_jobs = [

    _job

    for _job in JTBD_JOB_SCORES_Z.columns

    if _job not in _rankable_jobs
    and JTBD_JOB_SCORES_Z[_job]
        .notna()
        .sum() > 1
]


# ----------------------------------------------------------------------------
# 4. COMPLETE-CASE DONOR RANKING
#
# FIX:
# A donor enters the Top-N ranking ONLY if every rankable Job is measured.
#
# This ensures:
# - every donor is competing across the same Job set;
# - Local Stewardship missingness cannot count as "not Top-2";
# - missing Local cannot artificially make another Job easier to rank Top-2.
# ----------------------------------------------------------------------------

_measured_job_n = (

    JTBD_JOB_SCORES_Z[
        _rankable_jobs
    ]

    .notna()

    .sum(axis=1)
)


_ranking_eligible = (
    _measured_job_n
    .eq(len(_rankable_jobs))
)


JTBD_JOB_RANKS = (

    JTBD_JOB_SCORES_Z[
        _rankable_jobs
    ]

    .rank(
        axis=1,
        method="first",
        ascending=False,
    )
)


JTBD_TOPN_FLAGS = (

    JTBD_JOB_RANKS
    .le(JTBD_TOP_N)
    .where(
        _ranking_eligible,
        False,
    )
    .astype(bool)
)


JTBD_TOP1_FLAGS = (

    JTBD_JOB_RANKS
    .eq(1)
    .where(
        _ranking_eligible,
        False,
    )
    .astype(bool)
)


if _ranking_eligible.any():

    assert (

        JTBD_TOPN_FLAGS
        .loc[_ranking_eligible]
        .sum(axis=1)
        .eq(JTBD_TOP_N)
        .all()

    )


# ----------------------------------------------------------------------------
# 5. DONOR-LEVEL ASSIGNMENTS
# ----------------------------------------------------------------------------

JTBD_DONOR_ASSIGNMENTS = pd.DataFrame(
    index=JTBD_BASE.index
)

JTBD_DONOR_ASSIGNMENTS["cluster"] = (
    JTBD_BASE["cluster"]
)

JTBD_DONOR_ASSIGNMENTS[
    "jtbd_ranking_eligible"
] = _ranking_eligible

JTBD_DONOR_ASSIGNMENTS[
    "jtbd_jobs_measured"
] = _measured_job_n


for _r in range(
    1,
    JTBD_TOP_N + 1,
):

    JTBD_DONOR_ASSIGNMENTS[
        f"jtbd_top_{_r}"
    ] = JTBD_JOB_RANKS.apply(

        lambda row, r=_r:
            (
                row.index[
                    row.eq(r)
                ][0]
                if (
                    _ranking_eligible.loc[
                        row.name
                    ]
                    and row.eq(r).any()
                )
                else np.nan
            ),

        axis=1,
    )


JTBD_DONOR_ASSIGNMENTS[
    "jtbd_top_n"
] = (

    JTBD_DONOR_ASSIGNMENTS[
        [
            f"jtbd_top_{r}"
            for r in range(
                1,
                JTBD_TOP_N + 1,
            )
        ]
    ]

    .apply(
        lambda r:
            " | ".join(
                r.dropna()
                .astype(str)
            ),
        axis=1,
    )
)


# ----------------------------------------------------------------------------
# 6. CLUSTER-LEVEL TOP-N PREVALENCE
# ----------------------------------------------------------------------------

_overall_topn = (

    JTBD_TOPN_FLAGS
    .loc[_ranking_eligible]
    .mean(axis=0)
)


_overall_top1 = (

    JTBD_TOP1_FLAGS
    .loc[_ranking_eligible]
    .mean(axis=0)
)


_summary_rows = []


for _cluster in sorted(
    JTBD_BASE["cluster"]
    .dropna()
    .unique()
):

    _cluster_mask = (
        JTBD_BASE["cluster"]
        .eq(_cluster)
    )

    _eligible_mask = (
        _cluster_mask
        & _ranking_eligible
    )

    _cluster_n = int(
        _cluster_mask.sum()
    )

    _eligible_n = int(
        _eligible_mask.sum()
    )


    for _job in _rankable_jobs:

        _top_share = (

            float(
                JTBD_TOPN_FLAGS
                .loc[
                    _eligible_mask,
                    _job,
                ]
                .mean()
            )

            if _eligible_n

            else np.nan
        )


        _overall = float(
            _overall_topn[_job]
        )


        _evidence = (

            float(
                JTBD_JOB_SCORES_Z
                .loc[
                    _eligible_mask,
                    _job,
                ]
                .mean()
            )

            if _eligible_n

            else np.nan
        )


        _index = (

            _top_share / _overall

            if (
                pd.notna(_top_share)
                and _overall > 0
            )

            else np.nan
        )


        _standout = bool(

            pd.notna(_index)
            and pd.notna(_evidence)

            and _index
                >= JTBD_STANDOUT_MIN_INDEX

            and _evidence
                >= JTBD_STANDOUT_MIN_EVIDENCE_Z
        )


        _summary_rows.append({

            "cluster":
                int(_cluster),

            "cluster_n":
                _cluster_n,

            "ranking_eligible_n":
                _eligible_n,

            "ranking_eligible_share":
                (
                    _eligible_n / _cluster_n
                    if _cluster_n
                    else np.nan
                ),

            "Job":
                _job,

            f"top_{JTBD_TOP_N}_n":
                (
                    int(
                        JTBD_TOPN_FLAGS
                        .loc[
                            _eligible_mask,
                            _job,
                        ]
                        .sum()
                    )
                    if _eligible_n
                    else 0
                ),

            f"top_{JTBD_TOP_N}_share":
                _top_share,

            "overall_top_n_share":
                _overall,

            "index_vs_overall":
                _index,

            "top_1_share":
                (
                    float(
                        JTBD_TOP1_FLAGS
                        .loc[
                            _eligible_mask,
                            _job,
                        ]
                        .mean()
                    )
                    if _eligible_n
                    else np.nan
                ),

            "mean_behavioral_evidence_z":
                _evidence,

            "standout":
                _standout,

            "job_measured_share_in_cluster":
                float(
                    JTBD_JOB_SCORES_Z
                    .loc[
                        _cluster_mask,
                        _job,
                    ]
                    .notna()
                    .mean()
                ),
        })


JTBD_CLUSTER_SUMMARY = pd.DataFrame(
    _summary_rows
)


JTBD_STANDOUT_SUMMARY = (

    JTBD_CLUSTER_SUMMARY.loc[
        JTBD_CLUSTER_SUMMARY[
            "standout"
        ]
    ]

    .sort_values(
        [
            "cluster",
            "index_vs_overall",
        ],
        ascending=[
            True,
            False,
        ],
    )

    .reset_index(
        drop=True
    )
)


# ----------------------------------------------------------------------------
# 7. EVIDENCE-ONLY JOB SUMMARIES
#
# Tax Efficiency and Collective Participation live here.
# They do NOT affect donor Top-2 assignments.
# ----------------------------------------------------------------------------

_evidence_rows = []


for _cluster in sorted(
    JTBD_BASE["cluster"]
    .dropna()
    .unique()
):

    _cluster_mask = (
        JTBD_BASE["cluster"]
        .eq(_cluster)
    )


    for _job in _evidence_only_jobs:

        _x = JTBD_JOB_SCORES_Z.loc[
            _cluster_mask,
            _job,
        ]

        _evidence_rows.append({

            "cluster":
                int(_cluster),

            "Job":
                _job,

            "mean_behavioral_evidence_z":
                float(_x.mean()),

            "measured_share":
                float(_x.notna().mean()),
        })


JTBD_EVIDENCE_ONLY_SUMMARY = pd.DataFrame(
    _evidence_rows
)


# ----------------------------------------------------------------------------
# 8. BUSINESS-FRIENDLY OUTPUT
# ----------------------------------------------------------------------------

_n_eligible = int(
    _ranking_eligible.sum()
)

_n_total = len(
    _ranking_eligible
)


display(
    Markdown(
        f"""
### Jobs most reflected in observed behavior

**{len(_rankable_jobs)} Jobs currently enter the behavioral Top-{JTBD_TOP_N} ranking.**

For comparability, Top-{JTBD_TOP_N} rankings use only donors for whom **all {len(_rankable_jobs)} rankable Jobs are measurable**.

**{_n_eligible:,} of {_n_total:,} donors ({_n_eligible / _n_total:.0%})** meet that complete-case rule.

Each eligible donor contributes exactly **{JTBD_TOP_N} slots**.

A Job is labeled a **standout** for a cluster only when:

- Top-{JTBD_TOP_N} prevalence is at least **{JTBD_STANDOUT_MIN_INDEX:.2f}x overall**, and
- mean behavioral evidence is at least **+{JTBD_STANDOUT_MIN_EVIDENCE_Z:.2f} SD above average**.

These are behavioral signals consistent with Jobs — not direct evidence of donor motivation.
"""
    )
)


# -----------------------------------
# Scoring / coverage audit
# -----------------------------------

display(

    JTBD_JOB_AUDIT[
        [
            "Job",
            "Top-N status",
            "donor coverage",
            "usable features",
            "features used",
        ]
    ]

    .style

    .format({
        "donor coverage": "{:.0%}",
    })

    .hide(axis="index")

    .set_caption(
        "JTBD scoring audit — signed numbers are within-Job feature weights"
    )
)


# -----------------------------------
# Full cross-cluster Top-N matrix
# -----------------------------------

_top_col = (
    f"top_{JTBD_TOP_N}_share"
)


JTBD_TOPN_MATRIX = (

    JTBD_CLUSTER_SUMMARY

    .pivot(
        index="Job",
        columns="cluster",
        values=_top_col,
    )

    .reindex(
        _rankable_jobs
    )
)


JTBD_TOPN_MATRIX.columns = [

    f"Cluster {int(c)}"

    for c in JTBD_TOPN_MATRIX.columns
]


display(

    JTBD_TOPN_MATRIX

    .style

    .format("{:.0%}")

    .background_gradient(
        axis=None,
        cmap="Blues",
    )

    .set_caption(
        f"Share of complete-case donors in each cluster with the Job in behavioral Top {JTBD_TOP_N}"
    )
)


# -----------------------------------
# Cluster cards — standouts only
# -----------------------------------

for _cluster in sorted(
    JTBD_CLUSTER_SUMMARY[
        "cluster"
    ].unique()
):

    _all_cluster = (

        JTBD_CLUSTER_SUMMARY.loc[
            JTBD_CLUSTER_SUMMARY[
                "cluster"
            ].eq(_cluster)
        ]

        .copy()
    )


    _s = (

        _all_cluster.loc[
            _all_cluster[
                "standout"
            ]
        ]

        .sort_values(
            [
                "index_vs_overall",
                "mean_behavioral_evidence_z",
            ],
            ascending=[
                False,
                False,
            ],
        )

        .head(
            JTBD_SHOW_TOP_PER_CLUSTER
        )
    )


    _eligible_n = int(
        _all_cluster[
            "ranking_eligible_n"
        ].iloc[0]
    )

    _cluster_n = int(
        _all_cluster[
            "cluster_n"
        ].iloc[0]
    )


    if len(_s):

        _rows = []


        for _r in _s.itertuples(
            index=False
        ):

            _top_share = getattr(
                _r,
                _top_col,
            )

            _count = getattr(
                _r,
                f"top_{JTBD_TOP_N}_n",
            )


            _rows.append(

                "<tr>"

                f"<td style='padding:6px 8px;font-weight:600'>"
                f"{_r.Job}"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right'>"
                f"{_count:,} ({_top_share:.0%})"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right'>"
                f"{_r.overall_top_n_share:.0%}"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right;font-weight:600'>"
                f"{_r.index_vs_overall:.2f}x"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right;font-weight:600'>"
                f"{_r.mean_behavioral_evidence_z:+.2f} SD"
                f"</td>"

                "</tr>"
            )


        _body = f"""
        <table style="
            border-collapse:collapse;
            width:100%;
            font-size:12px;
        ">
        
            <thead>
                <tr style="background:#f8fafc">
        
                    <th style="text-align:left;padding:6px 8px">
                        Standout Job
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Cluster Top-{JTBD_TOP_N}
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Overall
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Index
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Evidence vs avg
                    </th>
        
                </tr>
            </thead>
        
            <tbody>
                {''.join(_rows)}
            </tbody>
        
        </table>
        """

    else:

        _body = (
            "<div style='color:#666;font-size:13px;padding:6px 0'>"
            "No Job clears both standout thresholds. "
            "See the full Top-N matrix for weaker relative signals."
            "</div>"
        )


    display(
        HTML(
            f"""
<div style="
    border:1px solid #d1d5db;
    border-radius:8px;
    padding:12px 14px;
    margin:12px 0 18px 0;
    background:white;
">

    <div style="
        font-size:19px;
        font-weight:700;
    ">
        Cluster {int(_cluster)} — standout behavioral Job signals
    </div>

    <div style="
        color:#666;
        font-size:12px;
        margin:3px 0 9px 0;
    ">
        {_eligible_n:,} of {_cluster_n:,} donors included in
        complete-case Top-{JTBD_TOP_N} ranking
    </div>

    {_body}

</div>
"""
        )
    )


# -----------------------------------
# Evidence-only Jobs
# -----------------------------------

if len(JTBD_EVIDENCE_ONLY_SUMMARY):

    JTBD_EVIDENCE_ONLY_MATRIX = (

        JTBD_EVIDENCE_ONLY_SUMMARY

        .pivot(
            index="Job",
            columns="cluster",
            values="mean_behavioral_evidence_z",
        )
    )

    JTBD_EVIDENCE_ONLY_MATRIX.columns = [

        f"Cluster {int(c)}"

        for c in JTBD_EVIDENCE_ONLY_MATRIX.columns
    ]


    display(
        Markdown(
            """
### Evidence-only Job signals

These are useful directional signals but **cannot receive donor Top-2 slots**.

This currently includes **Tax Efficiency** and **Collective Participation**.
"""
        )
    )


    display(

        JTBD_EVIDENCE_ONLY_MATRIX

        .style

        .format("{:+.2f} SD")

        .background_gradient(
            axis=None,
            cmap="Blues",
        )

        .set_caption(
            "Evidence-only Jobs — directional behavioral signals, not Job assignments"
        )
    )


# -----------------------------------
# Jobs with no defensible score
# -----------------------------------

_skipped_text = "<br>".join(

    f"<b>{job}</b>: {reason}"

    for job, reason
    in JTBD_NOT_RANKED.items()
)


display(

    HTML(
        """
<div style="
    font-size:12px;
    color:#555;
    margin-top:12px;
">
<b>Jobs without a defensible current behavioral score:</b><br>
"""
        + _skipped_text
        +
        """
<br><br>
<i>Interpretation guardrail:</i>
these scores describe observed behaviors consistent with Jobs,
not directly observed donor motivations.
</div>
"""
    )
)


print(
    "Objects created: "
    "JTBD_DONOR_ASSIGNMENTS, "
    "JTBD_JOB_SCORES_RAW, "
    "JTBD_JOB_SCORES_Z, "
    "JTBD_JOB_RANKS, "
    "JTBD_TOPN_FLAGS, "
    "JTBD_JOB_AUDIT, "
    "JTBD_CLUSTER_SUMMARY, "
    "JTBD_STANDOUT_SUMMARY, "
    "JTBD_TOPN_MATRIX, "
    "JTBD_EVIDENCE_ONLY_SUMMARY"
)


### Jobs most reflected in observed behavior

**9 Jobs currently enter the behavioral Top-2 ranking.**

For comparability, Top-2 rankings use only donors for whom **all 9 rankable Jobs are measurable**.

**41,490 of 54,631 donors (76%)** meet that complete-case rule.

Each eligible donor contributes exactly **2 slots**.

A Job is labeled a **standout** for a cluster only when:

- Top-2 prevalence is at least **1.25x overall**, and
- mean behavioral evidence is at least **+0.20 SD above average**.

These are behavioral signals consistent with Jobs — not direct evidence of donor motivation.


Job,Top-N status,donor coverage,usable features,features used
Concrete impact,RANKED,100%,3,median_gift_to_project_cost_ratio_24m (+0.75) | share_gifts_closed_project_24m (+1.00) | share_gifts_classroom_essentials_24m (+0.50)
Directed choice,RANKED,94%,2,share_gifts_with_prior_search_24m (+1.00) | project_page_pre_gift_mean_24m (+1.00)
Giving leverage,RANKED,100%,2,share_gifts_with_match_24m (+1.00) | mean_match_excess_24m (+1.00)
Tax efficiency,EVIDENCE ONLY,100%,3,share_gifts_daf_24m (+2.00) | share_gifts_year_end_final_week_24m (+1.00) | share_gifts_q4_24m (+0.50)
Sustained giving,RANKED,100%,5,n_active_months_24m (+1.00) | entropy_gift_month_norm_24m (+1.00) | is_continuing_donor_24m (+1.00) | tenure_days_any_giving (+0.75) | monthly_longest_streak_months (+0.50)
Confidence and risk reduction,RANKED,96%,3,share_gifts_repeat_school_24m (+1.00) | share_gifts_with_prior_search_24m (+0.75) | teacher_page_pre_gift_mean_24m (+0.50)
Relational support,RANKED,100%,4,share_gifts_repeat_teacher_24m (+1.00) | top_teacher_share_count_24m (+1.00) | gifts_per_teacher_24m (+0.75) | is_teacher_referred (+0.50)
Local stewardship,RANKED,81%,2,share_gifts_within_15mi_24m (+1.00) | median_distance_mi_24m (-1.00)
Collective participation,EVIDENCE ONLY,100%,4,sharing_active_months_24m (+1.00) | share_gifts_channel_sharetray_24m (+1.00) | share_gifts_channel_facebook_24m (+0.50) | share_gifts_channel_nextdoor_24m (+0.50)
Urgent action,RANKED,100%,5,days_gift_to_expiration_median_24m (-1.00) | share_gifts_expiring_soon_24m (+1.00) | share_gifts_late_cycle_24m (+0.50) | share_gifts_same_day_first_session_24m (+0.50) | days_first_session_to_gift_median_24m (-0.50)


,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6
Job,,,,,,
Concrete impact,17%,6%,76%,49%,5%,11%
Directed choice,17%,20%,19%,26%,24%,16%
Giving leverage,14%,10%,19%,7%,55%,21%
Sustained giving,48%,19%,27%,14%,8%,25%
Confidence and risk reduction,7%,34%,4%,14%,28%,25%
Relational support,4%,46%,2%,23%,35%,61%
Local stewardship,33%,38%,8%,35%,20%,0%
Urgent action,42%,9%,31%,21%,8%,15%
Values obligation,19%,19%,15%,11%,17%,26%


Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Sustained giving,"4,483 (48%)",25%,1.91x,+0.42 SD
Urgent action,"3,849 (42%)",22%,1.86x,+0.40 SD


Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Confidence and risk reduction,"2,960 (34%)",18%,1.88x,+0.69 SD
Relational support,"3,989 (46%)",26%,1.79x,+0.82 SD
Local stewardship,"3,311 (38%)",24%,1.58x,+0.81 SD


...

Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Giving leverage,"3,310 (55%)",20%,2.77x,+1.18 SD
Confidence and risk reduction,"1,702 (28%)",18%,1.56x,+0.69 SD
Relational support,"2,125 (35%)",26%,1.37x,+0.73 SD


Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Relational support,"2,699 (61%)",26%,2.37x,+0.92 SD
Values obligation,"1,166 (26%)",18%,1.48x,+0.24 SD
Confidence and risk reduction,"1,134 (25%)",18%,1.41x,+0.47 SD



### Evidence-only Job signals

These are useful directional signals but **cannot receive donor Top-2 slots**.

This currently includes **Tax Efficiency** and **Collective Participation**.


,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6
Job,,,,,,
Collective participation,-0.06 SD,+0.23 SD,-0.29 SD,-0.20 SD,+0.22 SD,+0.16 SD
Tax efficiency,+0.15 SD,-0.18 SD,+0.40 SD,-0.08 SD,-0.26 SD,-0.14 SD


Objects created: JTBD_DONOR_ASSIGNMENTS, JTBD_JOB_SCORES_RAW, JTBD_JOB_SCORES_Z, JTBD_JOB_RANKS, JTBD_TOPN_FLAGS, JTBD_JOB_AUDIT, JTBD_CLUSTER_SUMMARY, JTBD_STANDOUT_SUMMARY, JTBD_TOPN_MATRIX, JTBD_EVIDENCE_ONLY_SUMMARY


In [13]:
# ============================================================================
# EXECUTIVE READOUT ANALYTICAL LAYERS
# Run after the clustering + JTBD cells.
#
# Goal: collect the extended post-fit evidence needed for a CMO/CTO readout,
# using raw values + over/under-index vs the full clustered population wherever
# that comparison is meaningful. Nothing here changes the clustering fit.
# ============================================================================

from pathlib import Path
from html import escape
import re
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# -----------------------------------------------------------------------------
# 0. Guardrails + labels
# -----------------------------------------------------------------------------

_REQUIRED_OBJECTS = ["DONORS_WITH_CLUSTER", "ASSIGNMENTS", "UNITS_NOW"]
_missing_objects = [x for x in _REQUIRED_OBJECTS if x not in globals()]
if _missing_objects:
    raise NameError(
        "Run the clustering notebook through the selected K solution first. "
        f"Missing: {_missing_objects}"
    )

CLUSTER_LABELS = {
    1: "Loyal Local Regulars",
    2: "Steady Explorers",
    3: "Bursty Project Solvers",
    4: "Need-Responsive Finishers",
    5: "Local Loyal Campaign Responders",
    6: "Distant Relationship Loyalists",
}

# Reporting thresholds only; they do not affect any score or assignment.
EXEC_INDEX_OVER = 1.20
EXEC_INDEX_UNDER = 0.80
EXEC_CATEGORY_MIN_OVERALL_SHARE = 0.02
EXEC_CATEGORY_MIN_COVERAGE = 0.75
EXEC_JOB_PAIR_MIN_OVERALL_SHARE = 0.015
EXEC_TOP_CATEGORY_N = 4
EXEC_BOTTOM_CATEGORY_N = 2
EXEC_TOP_JOB_PAIR_N = 3

# -----------------------------------------------------------------------------
# 1. Pull the extra post-fit fields from the feature file when available
# -----------------------------------------------------------------------------

EXEC_BASE = DONORS_WITH_CLUSTER.copy()
if EXEC_BASE.index.name != "donor_id" and "donor_id" in EXEC_BASE.columns:
    EXEC_BASE = EXEC_BASE.set_index("donor_id")

# Explicit fields we want if they exist. Both current explicit _24m names and
# older legacy _12m names are supported. We never assume a legacy _12m field is
# literally a 12-month window; the availability audit below calls that out.
_EXPLICIT_WANTED = {
    # Value / economics
    "gift_amount_24m", "gift_amount_12m", "gift_amount_prev24m", "gift_amount_prev12m",
    "grand_amount_24m", "grand_amount_12m", "monthly_amount_24m", "monthly_amount_12m",
    "mean_gift_amount_24m", "mean_gift_amount_12m",
    "median_gift_amount_24m", "median_gift_amount_12m",
    "max_gift_amount_24m", "max_gift_amount_12m",
    "lifetime_amount", "lifetime_gift_count", "lifetime_max_gift_amount",
    "is_major_gift_donor",

    # Lifecycle / cadence
    "n_gifts_24m", "n_gifts_12m", "n_gifts_prev24m", "n_gifts_prev12m",
    "n_active_months_24m", "n_active_months_12m",
    "days_since_last_gift", "tenure_days", "tenure_days_any_giving",
    "is_new_donor_24m", "is_new_donor_12m",
    "is_reactivated_24m", "is_reactivated_12m",
    "is_continuing_donor_24m", "is_continuing_donor_12m",
    "is_monthly_donor_current",
    "monthly_active_months_24m", "monthly_active_months_12m",
    "monthly_longest_streak_months",
    "monthly_median_payment_amount_24m", "monthly_median_payment_amount_12m",
    "share_amount_monthly_24m", "share_amount_monthly_12m",
    "n_recurring_donation_ids", "months_since_first_monthly_join",

    # Site / product / decision journey
    "has_site_data_24m", "has_site_data_12m", "site_unit_is_session",
    "days_with_site_activity_24m", "days_with_site_activity_12m",
    "n_site_visits_24m", "n_site_visits_12m",
    "site_rows_24m", "site_rows_12m",
    "site_visits_per_active_day_24m", "site_visits_per_active_day_12m",
    "share_project_page_visits_24m", "share_project_page_visits_12m",
    "project_page_visits_day_total_24m", "project_page_visits_day_total_12m",
    "share_teacher_page_visits_24m", "share_teacher_page_visits_12m",
    "teacher_page_visits_day_total_24m", "teacher_page_visits_day_total_12m",
    "share_search_visits_24m", "share_search_visits_12m",
    "search_visits_day_total_24m", "search_visits_day_total_12m",
    "page_visits_per_active_day_24m", "page_visits_per_active_day_12m",
    "cart_visits_24m", "cart_visits_12m",
    "campaign_visit_share_24m", "campaign_visit_share_12m",
    "days_since_last_cart_visit",
    "project_page_pre_gift_mean_24m", "project_page_pre_gift_mean_12m",
    "teacher_page_pre_gift_mean_24m", "teacher_page_pre_gift_mean_12m",
    "search_pre_gift_mean_24m", "search_pre_gift_mean_12m",
    "cart_pre_gift_mean_24m", "cart_pre_gift_mean_12m",
    "site_days_pre_gift_mean_24m", "site_days_pre_gift_mean_12m",
    "share_gifts_with_prior_search_24m", "share_gifts_with_prior_search_12m",
    "share_gifts_same_day_first_session_24m", "share_gifts_same_day_first_session_12m",
    "days_first_session_to_gift_median_24m", "days_first_session_to_gift_median_12m",
    "decision_speed_measurable_24m", "decision_speed_measurable_12m",

    # Optional tip / platform support
    "avg_optional_donation_rate_24m", "avg_optional_donation_rate_12m",
    "share_gifts_with_optional_donation_24m", "share_gifts_with_optional_donation_12m",

    # Choice / need / category / geography
    "n_unique_categories_24m", "n_unique_categories_12m",
    "entropy_category_norm_24m", "entropy_category_norm_12m",
    "top_category_share_count_24m", "top_category_share_count_12m",
    "share_gifts_classroom_essentials_24m", "share_gifts_classroom_essentials_12m",
    "share_amount_classroom_essentials_24m", "share_amount_classroom_essentials_12m",
    "share_gifts_to_low_income_schools_24m", "share_gifts_to_low_income_schools_12m",
    "share_gifts_to_historically_underrepresented_race_schools_24m",
    "share_gifts_to_historically_underrepresented_race_schools_12m",
    "share_gifts_to_underserved_rural_schools_24m", "share_gifts_to_underserved_rural_schools_12m",
    "share_gifts_with_match_24m", "share_gifts_with_match_12m",
    "mean_match_excess_24m", "mean_match_excess_12m",
    "share_amount_with_match_24m", "share_amount_with_match_12m",
    "share_gifts_over_half_project_cost_24m", "share_gifts_over_half_project_cost_12m",
    "share_gifts_full_project_cost_24m", "share_gifts_full_project_cost_12m",
    "median_gift_to_project_cost_ratio_24m", "median_gift_to_project_cost_ratio_12m",
    "median_distance_mi_24m", "median_distance_mi_12m",
    "share_gifts_within_15mi_24m", "share_gifts_within_15mi_12m",
    "share_gifts_same_state_24m", "share_gifts_same_state_12m",

    # Context useful to CMO / CTO
    "is_teacher", "is_teacher_referred", "is_marketing_subscribed",
    "email_open_rate_24m", "email_open_rate_12m",
    "email_click_rate_24m", "email_click_rate_12m",
}

_csv_header = []
if "FEATURES_PATH" in globals() and Path(FEATURES_PATH).exists():
    _csv_header = pd.read_csv(FEATURES_PATH, nrows=0).columns.tolist()
    _header_set = set(_csv_header)

    # Dynamic families that are intentionally too wide to hard-code.
    _dynamic = {
        c for c in _csv_header
        if (
            c.startswith("share_count_category_")
            or c.startswith("share_amount_category_")
            or "session_duration" in c.lower()
            or ("duration" in c.lower() and ("session" in c.lower() or "visit" in c.lower()))
            or ("unique" in c.lower() and "project" in c.lower() and ("session" in c.lower() or "visit" in c.lower()))
            or "ltv" in c.lower()
            or "retention" in c.lower()
            or "retained" in c.lower()
            or "next_gift" in c.lower()
            or ("optional" in c.lower() and ("amount" in c.lower() or "dollar" in c.lower()))
        )
    }

    _wanted = (_EXPLICIT_WANTED | _dynamic) & _header_set
    _to_load = sorted([c for c in _wanted if c not in EXEC_BASE.columns and c != "donor_id"])

    if _to_load:
        _extra = pd.read_csv(FEATURES_PATH, usecols=["donor_id"] + _to_load).set_index("donor_id")
        EXEC_BASE = EXEC_BASE.join(_extra, how="left")
else:
    _header_set = set(EXEC_BASE.columns)


def _first_available(*names):
    for n in names:
        if n in EXEC_BASE.columns:
            return n
    return None


def _all_matching(predicate):
    return [c for c in EXEC_BASE.columns if predicate(c)]

# -----------------------------------------------------------------------------
# 2. Resolve canonical fields + derive a few clean executive metrics
# -----------------------------------------------------------------------------

F = {}

def _r(key, *names):
    F[key] = _first_available(*names)
    return F[key]

# Economic value
_r("recent_project_revenue", "gift_amount_24m", "gift_amount_12m")
_r("prior_project_revenue", "gift_amount_prev24m", "gift_amount_prev12m")
_r("recent_all_giving_revenue", "grand_amount_24m", "grand_amount_12m")
_r("recent_monthly_revenue", "monthly_amount_24m", "monthly_amount_12m")
_r("median_gift", "median_gift_amount_24m", "median_gift_amount_12m")
_r("mean_gift", "mean_gift_amount_24m", "mean_gift_amount_12m")
_r("max_gift", "max_gift_amount_24m", "max_gift_amount_12m")
_r("lifetime_revenue", "lifetime_amount")
_r("lifetime_gift_count", "lifetime_gift_count")
_r("major_gift", "is_major_gift_donor")

# Lifecycle
_r("tenure_any", "tenure_days_any_giving", "tenure_days")
_r("recency", "days_since_last_gift")
_r("active_months", "n_active_months_24m", "n_active_months_12m")
_r("new", "is_new_donor_24m", "is_new_donor_12m")
_r("reactivated", "is_reactivated_24m", "is_reactivated_12m")
_r("continuing", "is_continuing_donor_24m", "is_continuing_donor_12m")
_r("monthly_current", "is_monthly_donor_current")
_r("monthly_active_months", "monthly_active_months_24m", "monthly_active_months_12m")
_r("monthly_streak", "monthly_longest_streak_months")
_r("monthly_share", "share_amount_monthly_24m", "share_amount_monthly_12m")
_r("monthly_median_payment", "monthly_median_payment_amount_24m", "monthly_median_payment_amount_12m")

# Product / site
_r("has_site", "has_site_data_24m", "has_site_data_12m")
_r("site_days", "days_with_site_activity_24m", "days_with_site_activity_12m")
_r("site_visits", "n_site_visits_24m", "n_site_visits_12m")
_r("project_page_share", "share_project_page_visits_24m", "share_project_page_visits_12m")
_r("project_page_count", "project_page_visits_day_total_24m", "project_page_visits_day_total_12m")
_r("search_share", "share_search_visits_24m", "share_search_visits_12m")
_r("search_count", "search_visits_day_total_24m", "search_visits_day_total_12m")
_r("cart_count", "cart_visits_24m", "cart_visits_12m")
_r("page_visits_per_day", "page_visits_per_active_day_24m", "page_visits_per_active_day_12m")
_r("prior_search", "share_gifts_with_prior_search_24m", "share_gifts_with_prior_search_12m")
_r("same_day_decision", "share_gifts_same_day_first_session_24m", "share_gifts_same_day_first_session_12m")
_r("days_to_gift", "days_first_session_to_gift_median_24m", "days_first_session_to_gift_median_12m")
_r("decision_coverage", "decision_speed_measurable_24m", "decision_speed_measurable_12m")

# Tip
_r("tip_rate", "avg_optional_donation_rate_24m", "avg_optional_donation_rate_12m")
_r("tip_gift_share", "share_gifts_with_optional_donation_24m", "share_gifts_with_optional_donation_12m")

# Choice / need
_r("category_count", "n_unique_categories_24m", "n_unique_categories_12m")
_r("category_entropy", "entropy_category_norm_24m", "entropy_category_norm_12m")
_r("top_category_share", "top_category_share_count_24m", "top_category_share_count_12m")
_r("classroom_essentials", "share_gifts_classroom_essentials_24m", "share_gifts_classroom_essentials_12m")
_r("low_income", "share_gifts_to_low_income_schools_24m", "share_gifts_to_low_income_schools_12m")
_r("underrep_race", "share_gifts_to_historically_underrepresented_race_schools_24m", "share_gifts_to_historically_underrepresented_race_schools_12m")
_r("rural", "share_gifts_to_underserved_rural_schools_24m", "share_gifts_to_underserved_rural_schools_12m")
_r("match_share", "share_gifts_with_match_24m", "share_gifts_with_match_12m")
_r("match_excess", "mean_match_excess_24m", "mean_match_excess_12m")
_r("over_half", "share_gifts_over_half_project_cost_24m", "share_gifts_over_half_project_cost_12m")
_r("full_project", "share_gifts_full_project_cost_24m", "share_gifts_full_project_cost_12m")
_r("gift_project_ratio", "median_gift_to_project_cost_ratio_24m", "median_gift_to_project_cost_ratio_12m")
_r("local_share", "share_gifts_within_15mi_24m", "share_gifts_within_15mi_12m")
_r("distance", "median_distance_mi_24m", "median_distance_mi_12m")

# Fit components for a raw-value/index description of the actual geometry.
for _key, _names in {
    "repeat_teacher": ("share_gifts_repeat_teacher_24m", "share_gifts_repeat_teacher_12m"),
    "school_entropy": ("entropy_school_norm_24m", "entropy_school_norm_12m"),
    "month_entropy": ("entropy_gift_month_norm_24m", "entropy_gift_month_norm_12m"),
    "gifts_per_active_month": ("gifts_per_active_month_24m", "gifts_per_active_month_12m"),
    "modal_amount_share": ("modal_amount_share_24m", "modal_amount_share_12m"),
    "round_amount_share": ("share_gifts_round_amount_24m", "share_gifts_round_amount_12m"),
    "first_money": ("share_gifts_first_money_in_24m", "share_gifts_first_money_in_12m"),
    "closed_project": ("share_gifts_closed_project_24m", "share_gifts_closed_project_12m"),
    "big_event": ("share_gifts_big_event_24m", "share_gifts_big_event_12m"),
}.items():
    _r(_key, *_names)

# Derived donor-level platform-support states.
if F["tip_gift_share"]:
    _x = pd.to_numeric(EXEC_BASE[F["tip_gift_share"]], errors="coerce")
    EXEC_BASE["exec_always_opt_out"] = _x.le(1e-12).astype(float).where(_x.notna())
    EXEC_BASE["exec_always_tip"] = _x.ge(1 - 1e-12).astype(float).where(_x.notna())
    F["always_opt_out"] = "exec_always_opt_out"
    F["always_tip"] = "exec_always_tip"
else:
    F["always_opt_out"] = None
    F["always_tip"] = None

if F["site_days"]:
    _x = pd.to_numeric(EXEC_BASE[F["site_days"]], errors="coerce")
    EXEC_BASE["exec_any_site_activity"] = _x.gt(0).astype(float).where(_x.notna())
    F["any_site_activity"] = "exec_any_site_activity"
else:
    F["any_site_activity"] = None

# Dynamic fields that may not exist in the current feature build.
_ltv_fields = _all_matching(lambda c: "ltv" in c.lower())
_retention_outcome_fields = _all_matching(
    lambda c: any(t in c.lower() for t in ["retention", "retained", "next_gift"])
)
_session_duration_fields = _all_matching(
    lambda c: "duration" in c.lower() and ("session" in c.lower() or "visit" in c.lower())
)
_unique_project_session_fields = _all_matching(
    lambda c: "unique" in c.lower() and "project" in c.lower() and ("session" in c.lower() or "visit" in c.lower())
)
_tip_dollar_fields = _all_matching(
    lambda c: "optional" in c.lower() and ("amount" in c.lower() or "dollar" in c.lower()) and "rate" not in c.lower()
)

# -----------------------------------------------------------------------------
# 3. Metric registry: what we want to tell an executive
# -----------------------------------------------------------------------------

_METRICS = []

def add_metric(layer, label, key_or_field, agg="mean", fmt="number", note=""):
    field = F.get(key_or_field, key_or_field)
    if field and field in EXEC_BASE.columns:
        _METRICS.append({
            "layer": layer,
            "metric": label,
            "field": field,
            "agg": agg,
            "fmt": fmt,
            "note": note,
        })

# Core behaviors: the raw evidence underneath the five fit dimensions.
add_metric("Core behavior", "Repeat-teacher gift share", "repeat_teacher", "mean", "percent", "Higher = more relationship-loyal")
add_metric("Core behavior", "School breadth / entropy", "school_entropy", "mean", "percent", "Lower = more relationship-loyal")
add_metric("Core behavior", "Giving-month entropy", "month_entropy", "mean", "percent", "Higher = more distributed through time")
add_metric("Core behavior", "Gifts per active month", "gifts_per_active_month", "median", "number", "Higher = more concentrated / bursty")
add_metric("Core behavior", "Modal-amount gift share", "modal_amount_share", "mean", "percent", "Higher = more standardized amount pattern")
add_metric("Core behavior", "Round-amount gift share", "round_amount_share", "mean", "percent", "Higher = more standardized amount pattern")
add_metric("Core behavior", "Median gift / project-cost ratio", "gift_project_ratio", "median", "ratio", "Higher = more of the project need funded")
add_metric("Core behavior", "First-money-in gift share", "first_money", "mean", "percent", "Higher = more contributor / initiator behavior")
add_metric("Core behavior", "Closed-project gift share", "closed_project", "mean", "percent", "Fit input; final temporal semantics still require QA")
add_metric("Core behavior", "Gifts within 15 miles", "local_share", "mean", "percent", "Geography has 0.50 fit weight")
add_metric("Core behavior", "Matched-gift share", "match_share", "mean", "percent", "Trigger/campaign response")
add_metric("Core behavior", "Big-event gift share", "big_event", "mean", "percent", "Trigger/campaign response")

# Economic value
add_metric("Economic value", "24m project revenue per donor", "recent_project_revenue", "median", "dollar")
add_metric("Economic value", "Previous equal-length-window project revenue per donor", "prior_project_revenue", "median", "dollar", "Prior window matches the configured current-window length")
add_metric("Economic value", "24m all-giving revenue per donor", "recent_all_giving_revenue", "median", "dollar", "Project + monthly if available")
add_metric("Economic value", "Lifetime project revenue per donor", "lifetime_revenue", "median", "dollar", "Observed lifetime revenue, not predictive LTV")
add_metric("Economic value", "Lifetime project gifts", "lifetime_gift_count", "median", "number")
add_metric("Economic value", "Median project gift", "median_gift", "median", "dollar")
add_metric("Economic value", "Major-gift donor share", "major_gift", "mean", "percent")

# If a true predictive/model-derived LTV field exists in a newer feature build, include it.
if _ltv_fields:
    add_metric("Economic value", "Predictive LTV", _ltv_fields[0], "median", "dollar", "Model-derived only if source semantics are confirmed")

# Lifecycle + monthly
add_metric("Lifecycle + retention", "Giving tenure", "tenure_any", "median", "days")
add_metric("Lifecycle + retention", "Days since last project gift", "recency", "median", "days", "Lower = more recent")
add_metric("Lifecycle + retention", "Active giving months", "active_months", "median", "number")
add_metric("Lifecycle + retention", "New-donor share", "new", "mean", "percent")
add_metric("Lifecycle + retention", "Reactivated-donor share", "reactivated", "mean", "percent")
add_metric("Lifecycle + retention", "Continuing-donor share", "continuing", "mean", "percent")
add_metric("Lifecycle + retention", "Current monthly-donor share", "monthly_current", "mean", "percent")
add_metric("Lifecycle + retention", "Monthly active months", "monthly_active_months", "median", "number")
add_metric("Lifecycle + retention", "Longest monthly streak", "monthly_streak", "median", "number")
add_metric("Lifecycle + retention", "Share of giving from monthly", "monthly_share", "mean", "percent")
add_metric("Lifecycle + retention", "Median monthly payment", "monthly_median_payment", "median", "dollar")

# Product intensity
add_metric("Digital product intensity", "Site-source coverage", "has_site", "mean", "percent", "Measurement coverage; not product affinity")
add_metric("Digital product intensity", "Donors with any site activity", "any_site_activity", "mean", "percent")
add_metric("Digital product intensity", "Site-active days", "site_days", "median", "number")
add_metric("Digital product intensity", "Site visits", "site_visits", "median", "number")
add_metric("Digital product intensity", "Project-page visit share", "project_page_share", "mean", "percent")
add_metric("Digital product intensity", "Project-page visits", "project_page_count", "median", "number")
add_metric("Digital product intensity", "Search visit share", "search_share", "mean", "percent")
add_metric("Digital product intensity", "Search visits", "search_count", "mean", "number")
add_metric("Digital product intensity", "Cart visits", "cart_count", "median", "number")
add_metric("Digital product intensity", "Page visits / active day", "page_visits_per_day", "median", "number")
add_metric("Digital product intensity", "Gifts preceded by search", "prior_search", "mean", "percent", "Among measurable pre-gift site journeys")
add_metric("Digital product intensity", "Same-day first-session-to-gift share", "same_day_decision", "mean", "percent")
add_metric("Digital product intensity", "Days first session to gift", "days_to_gift", "median", "days", "Lower = faster observed decision path")

# Optional tip economics
add_metric("Optional tip economics", "Average optional-tip rate", "tip_rate", "mean", "percent")
add_metric("Optional tip economics", "Share of gifts with optional tip", "tip_gift_share", "mean", "percent")
add_metric("Optional tip economics", "Donors who always opt out", "always_opt_out", "mean", "percent")
add_metric("Optional tip economics", "Donors who tip on every observed gift", "always_tip", "mean", "percent")

# Choice + need affinity
add_metric("Choice + need affinity", "Unique categories supported", "category_count", "median", "number")
add_metric("Choice + need affinity", "Category breadth / entropy", "category_entropy", "mean", "percent")
add_metric("Choice + need affinity", "Top-category gift share", "top_category_share", "mean", "percent", "Higher = more category-focused")
add_metric("Choice + need affinity", "Classroom Essentials gift share", "classroom_essentials", "mean", "percent")
add_metric("Choice + need affinity", "Low-income-school gift share", "low_income", "mean", "percent")
add_metric("Choice + need affinity", "Historically underrepresented-race school share", "underrep_race", "mean", "percent")
add_metric("Choice + need affinity", "Underserved-rural school share", "rural", "mean", "percent")
add_metric("Choice + need affinity", "Matched-gift share", "match_share", "mean", "percent")
add_metric("Choice + need affinity", "Mean match excess", "match_excess", "mean", "ratio")
add_metric("Choice + need affinity", "Gifts >= half of project cost", "over_half", "mean", "percent")
add_metric("Choice + need affinity", "Gifts >= full project cost", "full_project", "mean", "percent")
add_metric("Choice + need affinity", "Median gift / project-cost ratio", "gift_project_ratio", "median", "ratio")

# Context that is often useful for interpreting actionability.
add_metric("Audience context", "Teachers", "is_teacher", "mean", "percent")
add_metric("Audience context", "Teacher-referred donors", "is_teacher_referred", "mean", "percent")
add_metric("Audience context", "Marketing subscribed", "is_marketing_subscribed", "mean", "percent")
add_metric("Audience context", "Median donor-to-school distance", "distance", "median", "miles")

# -----------------------------------------------------------------------------
# 4. Build the long cluster-vs-overall metric table
# -----------------------------------------------------------------------------

def _agg_value(s, agg):
    x = pd.to_numeric(s, errors="coerce").dropna()
    if not len(x):
        return np.nan
    if agg == "mean":
        return float(x.mean())
    if agg == "median":
        return float(x.median())
    if agg == "sum":
        return float(x.sum())
    raise ValueError(f"Unknown agg: {agg}")


def _safe_index(cluster_value, overall_value):
    if pd.isna(cluster_value) or pd.isna(overall_value) or abs(overall_value) < 1e-12:
        return np.nan
    # Ratio indexes are intended for non-negative descriptive measures.
    if cluster_value < 0 or overall_value < 0:
        return np.nan
    return float(cluster_value / overall_value)


_exec_rows = []
for spec in _METRICS:
    field = spec["field"]
    s_all = pd.to_numeric(EXEC_BASE[field], errors="coerce")
    overall_value = _agg_value(s_all, spec["agg"])
    overall_coverage = float(s_all.notna().mean())

    for cluster, sub in EXEC_BASE.groupby("cluster"):
        s = pd.to_numeric(sub[field], errors="coerce")
        cluster_value = _agg_value(s, spec["agg"])
        _exec_rows.append({
            "cluster": int(cluster),
            "cluster_label": CLUSTER_LABELS.get(int(cluster), f"Cluster {int(cluster)}"),
            "layer": spec["layer"],
            "metric": spec["metric"],
            "field": field,
            "aggregation": spec["agg"],
            "format": spec["fmt"],
            "note": spec["note"],
            "cluster_value": cluster_value,
            "overall_value": overall_value,
            "index_vs_overall": _safe_index(cluster_value, overall_value),
            "cluster_coverage": float(s.notna().mean()),
            "overall_coverage": overall_coverage,
        })

EXEC_PROFILE_LONG = pd.DataFrame(_exec_rows)

# -----------------------------------------------------------------------------
# 5. Revenue concentration: share of dollars vs share of donors
# -----------------------------------------------------------------------------

_cluster_sizes = EXEC_BASE["cluster"].value_counts().sort_index()
_donor_share = _cluster_sizes / len(EXEC_BASE)

_rev_candidates = [
    ("24m project revenue", F.get("recent_project_revenue")),
    ("24m all-giving revenue", F.get("recent_all_giving_revenue")),
    ("24m monthly revenue", F.get("recent_monthly_revenue")),
    ("Lifetime project revenue", F.get("lifetime_revenue")),
]

_rev_rows = []
for label, field in _rev_candidates:
    if not field or field not in EXEC_BASE.columns:
        continue
    x = pd.to_numeric(EXEC_BASE[field], errors="coerce")
    total = float(x.fillna(0).sum())
    if total <= 0:
        continue

    for cluster in sorted(_cluster_sizes.index):
        mask = EXEC_BASE["cluster"].eq(cluster)
        dollars = float(x.loc[mask].fillna(0).sum())
        revenue_share = dollars / total
        ds = float(_donor_share.loc[cluster])
        _rev_rows.append({
            "revenue_measure": label,
            "field": field,
            "cluster": int(cluster),
            "cluster_label": CLUSTER_LABELS.get(int(cluster), f"Cluster {int(cluster)}"),
            "donor_share": ds,
            "revenue_dollars": dollars,
            "revenue_share": revenue_share,
            "revenue_index_vs_donor_share": revenue_share / ds if ds > 0 else np.nan,
        })

EXEC_REVENUE_CONCENTRATION = pd.DataFrame(_rev_rows)

# -----------------------------------------------------------------------------
# 6. Gift-size distribution: robust quartiles by cluster
# -----------------------------------------------------------------------------

_dist_fields = [
    ("24m project revenue per donor", F.get("recent_project_revenue"), "dollar"),
    ("Lifetime project revenue per donor", F.get("lifetime_revenue"), "dollar"),
    ("Median project gift", F.get("median_gift"), "dollar"),
    ("Giving tenure", F.get("tenure_any"), "days"),
]

_dist_rows = []
for label, field, fmt in _dist_fields:
    if not field or field not in EXEC_BASE.columns:
        continue
    for cluster, sub in EXEC_BASE.groupby("cluster"):
        x = pd.to_numeric(sub[field], errors="coerce").dropna()
        if not len(x):
            continue
        _dist_rows.append({
            "cluster": int(cluster),
            "cluster_label": CLUSTER_LABELS.get(int(cluster), f"Cluster {int(cluster)}"),
            "metric": label,
            "field": field,
            "format": fmt,
            "p25": float(x.quantile(.25)),
            "p50": float(x.quantile(.50)),
            "p75": float(x.quantile(.75)),
        })
EXEC_DISTRIBUTIONS = pd.DataFrame(_dist_rows)

# -----------------------------------------------------------------------------
# 7. Category affinity: dynamic category-level over/under indexes
# -----------------------------------------------------------------------------

_category_fields = [
    c for c in EXEC_BASE.columns
    if c.startswith("share_count_category_")
    and not any(x in c.lower() for x in ["_other_", "_unknown_"])
]


def _category_label(field):
    x = re.sub(r"^share_count_category_", "", field)
    x = re.sub(r"_(24m|12m)$", "", x)
    return x.replace("_", " ").title()

_cat_rows = []
for field in _category_fields:
    x_all = pd.to_numeric(EXEC_BASE[field], errors="coerce")
    overall = float(x_all.mean()) if x_all.notna().any() else np.nan
    coverage = float(x_all.notna().mean())

    if pd.isna(overall) or overall < EXEC_CATEGORY_MIN_OVERALL_SHARE or coverage < EXEC_CATEGORY_MIN_COVERAGE:
        continue

    for cluster, sub in EXEC_BASE.groupby("cluster"):
        x = pd.to_numeric(sub[field], errors="coerce")
        value = float(x.mean()) if x.notna().any() else np.nan
        _cat_rows.append({
            "cluster": int(cluster),
            "cluster_label": CLUSTER_LABELS.get(int(cluster), f"Cluster {int(cluster)}"),
            "category": _category_label(field),
            "field": field,
            "cluster_share": value,
            "overall_share": overall,
            "index_vs_overall": _safe_index(value, overall),
            "coverage": coverage,
        })

EXEC_CATEGORY_INDEX_LONG = pd.DataFrame(_cat_rows)

# -----------------------------------------------------------------------------
# 8. Jobs: Top-2 prevalence indexes, portfolio view, evidence-only indexes,
#    and Job-pair combinations
# -----------------------------------------------------------------------------

if "JTBD_CLUSTER_SUMMARY" in globals() and len(JTBD_CLUSTER_SUMMARY):
    _top_col = f"top_{globals().get('JTBD_TOP_N', 2)}_share"
    EXEC_JOB_INDEX_LONG = JTBD_CLUSTER_SUMMARY.copy()
    EXEC_JOB_INDEX_LONG["cluster_label"] = EXEC_JOB_INDEX_LONG["cluster"].map(
        lambda c: CLUSTER_LABELS.get(int(c), f"Cluster {int(c)}")
    )

    _portfolio = []
    for job, sub in EXEC_JOB_INDEX_LONG.groupby("Job"):
        s = sub.dropna(subset=["index_vs_overall"]).sort_values("index_vs_overall")
        if not len(s):
            continue
        low = s.iloc[0]
        high = s.iloc[-1]
        _portfolio.append({
            "Job": job,
            "overall_top2_share": float(sub["overall_top_n_share"].dropna().iloc[0]) if sub["overall_top_n_share"].notna().any() else np.nan,
            "highest_cluster": CLUSTER_LABELS.get(int(high["cluster"]), f"Cluster {int(high['cluster'])}"),
            "highest_index": float(high["index_vs_overall"]),
            "highest_cluster_top2_share": float(high[_top_col]),
            "lowest_cluster": CLUSTER_LABELS.get(int(low["cluster"]), f"Cluster {int(low['cluster'])}"),
            "lowest_index": float(low["index_vs_overall"]),
            "spread_high_minus_low": float(high["index_vs_overall"] - low["index_vs_overall"]),
        })
    EXEC_JOB_PORTFOLIO = pd.DataFrame(_portfolio).sort_values("spread_high_minus_low", ascending=False)
else:
    EXEC_JOB_INDEX_LONG = pd.DataFrame()
    EXEC_JOB_PORTFOLIO = pd.DataFrame()

if not EXEC_JOB_INDEX_LONG.empty and "ranking_eligible_share" in EXEC_JOB_INDEX_LONG.columns:
    EXEC_JOB_COVERAGE = (
        EXEC_JOB_INDEX_LONG[["cluster", "cluster_label", "cluster_n", "ranking_eligible_n", "ranking_eligible_share"]]
        .drop_duplicates("cluster")
        .sort_values("cluster")
        .reset_index(drop=True)
    )
else:
    EXEC_JOB_COVERAGE = pd.DataFrame()

# Evidence-only Jobs: use the raw 0-1 behavioral evidence score, expressed as
# cluster mean / overall mean rather than +/- SD.
_evidence_job_names = []
if "JTBD_JOB_AUDIT" in globals() and len(JTBD_JOB_AUDIT):
    _evidence_job_names = JTBD_JOB_AUDIT.loc[
        JTBD_JOB_AUDIT["Top-N status"].astype(str).str.startswith("EVIDENCE ONLY"), "Job"
    ].tolist()

_evidence_rows = []
if "JTBD_JOB_SCORES_RAW" in globals() and len(JTBD_JOB_SCORES_RAW):
    for job in _evidence_job_names:
        if job not in JTBD_JOB_SCORES_RAW.columns:
            continue
        x = pd.to_numeric(JTBD_JOB_SCORES_RAW[job], errors="coerce")
        overall = float(x.mean()) if x.notna().any() else np.nan
        for cluster in sorted(EXEC_BASE["cluster"].unique()):
            ids = EXEC_BASE.index[EXEC_BASE["cluster"].eq(cluster)]
            value = float(x.reindex(ids).mean())
            _evidence_rows.append({
                "Job": job,
                "cluster": int(cluster),
                "cluster_label": CLUSTER_LABELS.get(int(cluster), f"Cluster {int(cluster)}"),
                "cluster_raw_score": value,
                "overall_raw_score": overall,
                "index_vs_overall": _safe_index(value, overall),
                "measured_share": float(x.reindex(ids).notna().mean()),
            })
EXEC_EVIDENCE_ONLY_JOB_INDEX = pd.DataFrame(_evidence_rows)

# Job pairs: useful for sequencing / combined propositions.
_pair_rows = []
if "JTBD_DONOR_ASSIGNMENTS" in globals() and len(JTBD_DONOR_ASSIGNMENTS):
    _jd = JTBD_DONOR_ASSIGNMENTS.copy()
    if "jtbd_ranking_eligible" in _jd.columns:
        _jd = _jd.loc[_jd["jtbd_ranking_eligible"]].copy()

    c1, c2 = "jtbd_top_1", "jtbd_top_2"
    if c1 in _jd.columns and c2 in _jd.columns:
        _jd = _jd.dropna(subset=[c1, c2]).copy()
        _jd["Job pair"] = _jd[[c1, c2]].apply(lambda r: " + ".join(sorted([str(r[c1]), str(r[c2])])), axis=1)

        _overall_pair = _jd["Job pair"].value_counts(normalize=True)
        for cluster, sub in _jd.groupby("cluster"):
            _cluster_pair = sub["Job pair"].value_counts(normalize=True)
            for pair, overall_share in _overall_pair.items():
                if overall_share < EXEC_JOB_PAIR_MIN_OVERALL_SHARE:
                    continue
                cluster_share = float(_cluster_pair.get(pair, 0.0))
                _pair_rows.append({
                    "cluster": int(cluster),
                    "cluster_label": CLUSTER_LABELS.get(int(cluster), f"Cluster {int(cluster)}"),
                    "Job pair": pair,
                    "cluster_share": cluster_share,
                    "overall_share": float(overall_share),
                    "index_vs_overall": cluster_share / overall_share if overall_share > 0 else np.nan,
                })
EXEC_JOB_PAIR_INDEX_LONG = pd.DataFrame(_pair_rows)

# -----------------------------------------------------------------------------
# 9. Availability audit: explicitly show what the current feature build cannot
#    answer yet so the executive report does not silently overclaim.
# -----------------------------------------------------------------------------

_availability = [
    ("Economic value", "Observed lifetime revenue", F.get("lifetime_revenue"), "Available lifetime project-gift revenue; this is not predictive LTV."),
    ("Economic value", "Predictive LTV", _ltv_fields[0] if _ltv_fields else None, "Requires a true LTV/predicted future-value field or separate model."),
    ("Economic value", "Current-window project revenue", F.get("recent_project_revenue"), "Current notebook window is 24 months."),
    ("Economic value", "Current-window all-giving revenue", F.get("recent_all_giving_revenue"), "Preferred if project + monthly dollars are available."),
    ("Lifecycle + retention", "Current lifecycle state", F.get("continuing") or F.get("reactivated") or F.get("new"), "New/reactivated/continuing are current-window state descriptors."),
    ("Lifecycle + retention", "Future retention / next-gift outcome", _retention_outcome_fields[0] if _retention_outcome_fields else None, "Needs a subsequent observation window; should remain post-fit."),
    ("Digital product intensity", "Site-active days", F.get("site_days"), "Available if site source is observed."),
    ("Digital product intensity", "True-session grain", "site_unit_is_session" if "site_unit_is_session" in EXEC_BASE.columns else None, "If 0, visit counts are donor-day rather than true sessions."),
    ("Digital product intensity", "Project-page activity", F.get("project_page_count") or F.get("project_page_share"), "Available in current site feature family if loaded."),
    ("Digital product intensity", "Unique projects viewed per session", _unique_project_session_fields[0] if _unique_project_session_fields else None, "Not present in the current dictionary unless a newer feature build added it."),
    ("Digital product intensity", "Session duration", _session_duration_fields[0] if _session_duration_fields else None, "Current site source may be donor-day rather than true session grain."),
    ("Optional tip economics", "Tip rate + gift participation", F.get("tip_rate") or F.get("tip_gift_share"), "Available as rates/shares."),
    ("Optional tip economics", "Tip dollars / revenue contribution", _tip_dollar_fields[0] if _tip_dollar_fields else None, "Do not infer dollars from the rate unless source semantics are explicitly validated."),
    ("Choice + need affinity", "Category-level affinity", _category_fields[0] if _category_fields else None, "Dynamic category share fields; rare other/unknown buckets excluded from executive standouts."),
    ("Jobs", "Top-2 Job prevalence + indexes", "JTBD_CLUSTER_SUMMARY" if "JTBD_CLUSTER_SUMMARY" in globals() else None, "Behavioral evidence consistent with Jobs, not observed motivation."),
    ("Jobs", "Job-pair combinations", "JTBD_DONOR_ASSIGNMENTS" if "JTBD_DONOR_ASSIGNMENTS" in globals() else None, "Useful for sequencing multiple Jobs in journeys/messages."),
]

_av_rows = []
for layer, need, field, note in _availability:
    coverage = np.nan
    if field and field in EXEC_BASE.columns:
        coverage = float(pd.to_numeric(EXEC_BASE[field], errors="coerce").notna().mean())
    _av_rows.append({
        "layer": layer,
        "needed": need,
        "status": "AVAILABLE" if field else "MISSING / FOLLOW-ON",
        "field_or_object": field or "-",
        "coverage": coverage,
        "note": note,
    })
EXEC_AVAILABILITY_AUDIT = pd.DataFrame(_av_rows)

# -----------------------------------------------------------------------------
# 10. HTML helpers
# -----------------------------------------------------------------------------

def _fmt(v, fmt):
    if pd.isna(v):
        return "-"
    if fmt == "percent":
        return f"{v:.0%}"
    if fmt == "dollar":
        return f"${v:,.0f}"
    if fmt == "days":
        return f"{v:,.0f} days"
    if fmt == "miles":
        return f"{v:,.0f} mi"
    if fmt == "ratio":
        return f"{v:.2f}x"
    return f"{v:,.1f}"


def _idx_badge(v):
    if pd.isna(v):
        return "<span style='color:#888'>n/a</span>"
    if v >= EXEC_INDEX_OVER:
        bg, tag = "#dbeafe", "OVER"
    elif v <= EXEC_INDEX_UNDER:
        bg, tag = "#fef3c7", "UNDER"
    else:
        bg, tag = "#f3f4f6", ""
    return (
        f"<span style='display:inline-block;padding:2px 6px;border-radius:4px;background:{bg};font-weight:700'>"
        f"{v:.2f}x{' ' + tag if tag else ''}</span>"
    )


def _title(text, sub=None):
    html = f"<div style='font-size:22px;font-weight:750;margin:24px 0 4px 0'>{escape(text)}</div>"
    if sub:
        html += f"<div style='font-size:13px;color:#666;margin-bottom:10px'>{escape(sub)}</div>"
    display(HTML(html))


def _metric_matrix(layer):
    d = EXEC_PROFILE_LONG.loc[EXEC_PROFILE_LONG["layer"].eq(layer)].copy()
    if d.empty:
        return "<div style='color:#777'>No available metrics in this layer.</div>"

    metric_order = list(dict.fromkeys(d["metric"].tolist()))
    clusters = sorted(d["cluster"].unique())
    rows = []
    for metric in metric_order:
        m = d.loc[d["metric"].eq(metric)]
        r0 = m.iloc[0]
        cells = [
            f"<td style='padding:6px 8px;font-weight:650'>{escape(metric)}"
            + (f"<div style='font-size:10px;color:#777;font-weight:400'>{escape(r0['note'])}</div>" if r0["note"] else "")
            + "</td>",
            f"<td style='padding:6px 8px;text-align:right;color:#555'><div>{_fmt(r0['overall_value'], r0['format'])}</div>" + (f"<div style='font-size:9px;color:#888'>{r0['overall_coverage']:.0%} measured</div>" if r0['overall_coverage'] < .95 else "") + "</td>",
        ]
        for c in clusters:
            rr = m.loc[m["cluster"].eq(c)].iloc[0]
            cells.append(
                "<td style='padding:6px 8px;text-align:right;white-space:nowrap'>"
                f"<div style='font-weight:650'>{_fmt(rr['cluster_value'], rr['format'])}</div>"
                f"<div style='margin-top:2px'>{_idx_badge(rr['index_vs_overall'])}</div>"
                + (f"<div style='font-size:9px;color:#888'>{rr['cluster_coverage']:.0%} measured</div>" if rr['cluster_coverage'] < .95 else "")
                + "</td>"
            )
        rows.append("<tr style='border-bottom:1px solid #eee'>" + "".join(cells) + "</tr>")

    heads = "".join(
        f"<th style='padding:6px 8px;text-align:right'>{c}<div style='font-size:10px;font-weight:400'>{escape(CLUSTER_LABELS.get(c, ''))}</div></th>"
        for c in clusters
    )
    return (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse;width:100%;font-size:12px'>"
        "<thead><tr style='background:#f8fafc;border-bottom:1px solid #ccc'>"
        "<th style='padding:6px 8px;text-align:left'>Metric</th>"
        "<th style='padding:6px 8px;text-align:right'>Overall</th>"
        + heads + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"
    )


def _job_matrix_html():
    if EXEC_JOB_INDEX_LONG.empty:
        return "<div style='color:#777'>Run the JTBD cell first.</div>"
    top_col = f"top_{globals().get('JTBD_TOP_N', 2)}_share"
    jobs = EXEC_JOB_INDEX_LONG.groupby("Job")["overall_top_n_share"].first().sort_values(ascending=False).index.tolist()
    clusters = sorted(EXEC_JOB_INDEX_LONG["cluster"].unique())
    rows = []
    for job in jobs:
        sub = EXEC_JOB_INDEX_LONG.loc[EXEC_JOB_INDEX_LONG["Job"].eq(job)]
        overall = sub["overall_top_n_share"].dropna().iloc[0]
        cells = [
            f"<td style='padding:6px 8px;font-weight:650'>{escape(job)}</td>",
            f"<td style='padding:6px 8px;text-align:right'>{overall:.0%}</td>",
        ]
        for c in clusters:
            r = sub.loc[sub["cluster"].eq(c)].iloc[0]
            cells.append(
                "<td style='padding:6px 8px;text-align:right;white-space:nowrap'>"
                f"<div style='font-weight:650'>{r[top_col]:.0%}</div>"
                f"<div>{_idx_badge(r['index_vs_overall'])}</div>"
                "</td>"
            )
        rows.append("<tr style='border-bottom:1px solid #eee'>" + "".join(cells) + "</tr>")

    heads = "".join(
        f"<th style='padding:6px 8px;text-align:right'>C{c}<div style='font-size:10px;font-weight:400'>{escape(CLUSTER_LABELS.get(c, ''))}</div></th>"
        for c in clusters
    )
    return (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse;width:100%;font-size:12px'>"
        "<thead><tr style='background:#f8fafc;border-bottom:1px solid #ccc'>"
        "<th style='padding:6px 8px;text-align:left'>Behavioral Job signal</th>"
        "<th style='padding:6px 8px;text-align:right'>Overall Top-2</th>"
        + heads + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"
    )


def _revenue_html():
    if EXEC_REVENUE_CONCENTRATION.empty:
        return "<div style='color:#777'>No revenue fields available.</div>"
    rows = []
    for measure, sub in EXEC_REVENUE_CONCENTRATION.groupby("revenue_measure", sort=False):
        for r in sub.itertuples(index=False):
            rows.append(
                "<tr style='border-bottom:1px solid #eee'>"
                f"<td style='padding:6px 8px;font-weight:650'>{escape(measure)}</td>"
                f"<td style='padding:6px 8px'>C{r.cluster} {escape(r.cluster_label)}</td>"
                f"<td style='padding:6px 8px;text-align:right'>{r.donor_share:.1%}</td>"
                f"<td style='padding:6px 8px;text-align:right'>${r.revenue_dollars:,.0f}</td>"
                f"<td style='padding:6px 8px;text-align:right'>{r.revenue_share:.1%}</td>"
                f"<td style='padding:6px 8px;text-align:right'>{_idx_badge(r.revenue_index_vs_donor_share)}</td>"
                "</tr>"
            )
    return (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse;width:100%;font-size:12px'>"
        "<thead><tr style='background:#f8fafc;border-bottom:1px solid #ccc'>"
        "<th style='padding:6px 8px;text-align:left'>Revenue measure</th>"
        "<th style='padding:6px 8px;text-align:left'>Cluster</th>"
        "<th style='padding:6px 8px;text-align:right'>Donor share</th>"
        "<th style='padding:6px 8px;text-align:right'>Dollars</th>"
        "<th style='padding:6px 8px;text-align:right'>Revenue share</th>"
        "<th style='padding:6px 8px;text-align:right'>Revenue index</th>"
        "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"
    )



def _distribution_html():
    if EXEC_DISTRIBUTIONS.empty:
        return "<div style='color:#777'>No robust distribution fields available.</div>"
    rows = []
    for metric, sub in EXEC_DISTRIBUTIONS.groupby("metric", sort=False):
        for _, r in sub.sort_values("cluster").iterrows():
            fmt = r["format"]
            rows.append(
                "<tr style='border-bottom:1px solid #eee'>"
                f"<td style='padding:5px 7px;font-weight:650'>{escape(metric)}</td>"
                f"<td style='padding:5px 7px'>C{int(r['cluster'])} {escape(r['cluster_label'])}</td>"
                f"<td style='padding:5px 7px;text-align:right'>{_fmt(r['p25'], fmt)}</td>"
                f"<td style='padding:5px 7px;text-align:right;font-weight:650'>{_fmt(r['p50'], fmt)}</td>"
                f"<td style='padding:5px 7px;text-align:right'>{_fmt(r['p75'], fmt)}</td>"
                "</tr>"
            )
    return (
        "<div style='margin-top:12px;font-weight:700'>Distribution context (P25 / median / P75)</div>"
        "<div style='overflow-x:auto'><table style='border-collapse:collapse;width:100%;font-size:11px;margin-top:5px'>"
        "<tr style='background:#f8fafc'><th style='text-align:left;padding:5px 7px'>Metric</th>"
        "<th style='text-align:left;padding:5px 7px'>Cluster</th><th style='text-align:right;padding:5px 7px'>P25</th>"
        "<th style='text-align:right;padding:5px 7px'>Median</th><th style='text-align:right;padding:5px 7px'>P75</th></tr>"
        + "".join(rows) + "</table></div></div>"
    )

def _category_cards_html():
    if EXEC_CATEGORY_INDEX_LONG.empty:
        return "<div style='color:#777'>No category-level fields met the display thresholds.</div>"
    cards = []
    for c in sorted(EXEC_CATEGORY_INDEX_LONG["cluster"].unique()):
        sub = EXEC_CATEGORY_INDEX_LONG.loc[EXEC_CATEGORY_INDEX_LONG["cluster"].eq(c)].copy()
        over = sub.sort_values("index_vs_overall", ascending=False).head(EXEC_TOP_CATEGORY_N)
        under = sub.sort_values("index_vs_overall", ascending=True).head(EXEC_BOTTOM_CATEGORY_N)
        show = pd.concat([over, under]).drop_duplicates("category")
        rows = []
        for r in show.itertuples(index=False):
            rows.append(
                "<tr>"
                f"<td style='padding:4px 6px'>{escape(r.category)}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{r.cluster_share:.0%}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{r.overall_share:.0%}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{_idx_badge(r.index_vs_overall)}</td>"
                "</tr>"
            )
        cards.append(
            "<div style='border:1px solid #ddd;border-radius:7px;padding:10px 12px;margin:6px;min-width:320px;flex:1'>"
            f"<div style='font-size:15px;font-weight:700'>C{c} {escape(CLUSTER_LABELS.get(c, ''))}</div>"
            "<table style='width:100%;font-size:11px;border-collapse:collapse;margin-top:5px'>"
            "<tr style='background:#f8fafc'><th style='text-align:left;padding:4px 6px'>Category</th>"
            "<th style='text-align:right;padding:4px 6px'>Cluster</th><th style='text-align:right;padding:4px 6px'>Overall</th>"
            "<th style='text-align:right;padding:4px 6px'>Index</th></tr>"
            + "".join(rows) + "</table></div>"
        )
    return "<div style='display:flex;flex-wrap:wrap'>" + "".join(cards) + "</div>"


def _job_pair_cards_html():
    if EXEC_JOB_PAIR_INDEX_LONG.empty:
        return "<div style='color:#777'>Job-pair output unavailable; run the JTBD ranking cell first.</div>"
    cards = []
    for c in sorted(EXEC_JOB_PAIR_INDEX_LONG["cluster"].unique()):
        sub = EXEC_JOB_PAIR_INDEX_LONG.loc[EXEC_JOB_PAIR_INDEX_LONG["cluster"].eq(c)].copy()
        sub = sub.sort_values(["index_vs_overall", "cluster_share"], ascending=[False, False]).head(EXEC_TOP_JOB_PAIR_N)
        rows = []
        for _, r in sub.iterrows():
            rows.append(
                "<tr>"
                f"<td style='padding:4px 6px'>{escape(str(r['Job pair']))}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{r['cluster_share']:.0%}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{r['overall_share']:.0%}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{_idx_badge(r['index_vs_overall'])}</td>"
                "</tr>"
            )
        cards.append(
            "<div style='border:1px solid #ddd;border-radius:7px;padding:10px 12px;margin:6px;min-width:360px;flex:1'>"
            f"<div style='font-size:15px;font-weight:700'>C{c} {escape(CLUSTER_LABELS.get(c, ''))}</div>"
            "<table style='width:100%;font-size:11px;border-collapse:collapse;margin-top:5px'>"
            "<tr style='background:#f8fafc'><th style='text-align:left;padding:4px 6px'>Over-indexed Job pair</th>"
            "<th style='text-align:right;padding:4px 6px'>Cluster</th><th style='text-align:right;padding:4px 6px'>Overall</th>"
            "<th style='text-align:right;padding:4px 6px'>Index</th></tr>"
            + "".join(rows) + "</table></div>"
        )
    return "<div style='display:flex;flex-wrap:wrap'>" + "".join(cards) + "</div>"

def _availability_html():
    rows = []
    for r in EXEC_AVAILABILITY_AUDIT.itertuples(index=False):
        status_bg = "#dcfce7" if r.status == "AVAILABLE" else "#fef3c7"
        cov = "-" if pd.isna(r.coverage) else f"{r.coverage:.0%}"
        rows.append(
            "<tr style='border-bottom:1px solid #eee'>"
            f"<td style='padding:5px 7px'>{escape(r.layer)}</td>"
            f"<td style='padding:5px 7px;font-weight:650'>{escape(r.needed)}</td>"
            f"<td style='padding:5px 7px'><span style='background:{status_bg};padding:2px 5px;border-radius:4px;font-weight:700'>{escape(r.status)}</span></td>"
            f"<td style='padding:5px 7px;font-family:ui-monospace,monospace;font-size:10px'>{escape(str(r.field_or_object))}</td>"
            f"<td style='padding:5px 7px;text-align:right'>{cov}</td>"
            f"<td style='padding:5px 7px;color:#555'>{escape(r.note)}</td>"
            "</tr>"
        )
    return (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse;width:100%;font-size:11px'>"
        "<thead><tr style='background:#f8fafc'><th style='text-align:left;padding:5px 7px'>Layer</th>"
        "<th style='text-align:left;padding:5px 7px'>Need</th><th style='text-align:left;padding:5px 7px'>Status</th>"
        "<th style='text-align:left;padding:5px 7px'>Field / object</th><th style='text-align:right;padding:5px 7px'>Coverage</th>"
        "<th style='text-align:left;padding:5px 7px'>Interpretation</th></tr></thead><tbody>"
        + "".join(rows) + "</tbody></table></div>"
    )

# -----------------------------------------------------------------------------
# 11. Render the executive analysis in a readable sequence
# -----------------------------------------------------------------------------

_display_weight = globals().get("UNIT_WEIGHTS", {}).get("jtbd_local_stewardship", 1.0)
_title(
    "Executive-readout analytical layers",
    f"Selected K={globals().get('SELECTED_K', '?')} | local stewardship fit weight={_display_weight:.2f} | {len(EXEC_BASE):,} donors. Cells show raw value + index vs overall; index is descriptive, not good/bad."
)

_title("A. What the current data can answer", "Missing items are surfaced rather than inferred.")
display(HTML(_availability_html()))

_title("B. Economic value", "Revenue concentration answers whether a cluster carries more or less revenue than its population size would suggest.")
display(HTML(_metric_matrix("Economic value")))
display(HTML("<div style='height:10px'></div>" + _revenue_html()))
display(HTML(_distribution_html()))

_title("C. Lifecycle + monthly relationship", "These are current/lifetime descriptors; true future retention still requires a later observation window.")
display(HTML(_metric_matrix("Lifecycle + retention")))

_title("D. Digital product intensity", "Site coverage and journey observability matter; read the raw values together with coverage, not as pure motivation.")
display(HTML(_metric_matrix("Digital product intensity")))

_title("E. Optional tip economics", "Platform support is profiled after clustering so it can be economically important without defining persona identity.")
display(HTML(_metric_matrix("Optional tip economics")))

_title("F. Choice + need affinity", "Broad merchandising/recommendation signals first; detailed categories follow below.")
display(HTML(_metric_matrix("Choice + need affinity")))

_title("G. Category affinities", f"Shows categories with >= {EXEC_CATEGORY_MIN_OVERALL_SHARE:.0%} overall average gift-share baseline; rare other/unknown buckets excluded.")
display(HTML(_category_cards_html()))

_title("H. Jobs across the portfolio", "Each cell is cluster Top-2 prevalence plus its index vs the overall donor population. This is behavioral evidence consistent with a Job, not stated motivation.")
display(HTML(_job_matrix_html()))

if not EXEC_JOB_COVERAGE.empty:
    _rows = []
    for _, r in EXEC_JOB_COVERAGE.iterrows():
        _rows.append(
            "<tr style='border-bottom:1px solid #eee'>"
            f"<td style='padding:4px 6px'>C{int(r['cluster'])} {escape(r['cluster_label'])}</td>"
            f"<td style='padding:4px 6px;text-align:right'>{int(r['ranking_eligible_n']):,}</td>"
            f"<td style='padding:4px 6px;text-align:right;font-weight:650'>{r['ranking_eligible_share']:.0%}</td>"
            "</tr>"
        )
    display(HTML(
        "<div style='margin-top:10px;font-weight:700'>Job-ranking coverage</div>"
        "<div style='font-size:11px;color:#666;margin-bottom:4px'>Top-2 comparisons use donors with all rankable Jobs measured; large coverage differences by cluster would weaken direct comparisons.</div>"
        "<table style='border-collapse:collapse;font-size:11px;min-width:520px'>"
        "<tr style='background:#f8fafc'><th style='text-align:left;padding:4px 6px'>Cluster</th>"
        "<th style='text-align:right;padding:4px 6px'>Eligible donors</th><th style='text-align:right;padding:4px 6px'>Eligible share</th></tr>"
        + "".join(_rows) + "</table>"
    ))

if not EXEC_JOB_PORTFOLIO.empty:
    _jp = EXEC_JOB_PORTFOLIO.copy()
    _rows = []
    for r in _jp.itertuples(index=False):
        _rows.append(
            "<tr style='border-bottom:1px solid #eee'>"
            f"<td style='padding:5px 7px;font-weight:650'>{escape(r.Job)}</td>"
            f"<td style='padding:5px 7px;text-align:right'>{r.overall_top2_share:.0%}</td>"
            f"<td style='padding:5px 7px'>{escape(r.highest_cluster)}</td>"
            f"<td style='padding:5px 7px;text-align:right'>{_idx_badge(r.highest_index)}</td>"
            f"<td style='padding:5px 7px'>{escape(r.lowest_cluster)}</td>"
            f"<td style='padding:5px 7px;text-align:right'>{_idx_badge(r.lowest_index)}</td>"
            "</tr>"
        )
    display(HTML(
        "<div style='margin-top:12px;font-weight:700'>Which Jobs actually separate the portfolio?</div>"
        "<table style='border-collapse:collapse;width:100%;font-size:11px;margin-top:5px'>"
        "<tr style='background:#f8fafc'><th style='text-align:left;padding:5px 7px'>Job</th>"
        "<th style='text-align:right;padding:5px 7px'>Overall Top-2</th><th style='text-align:left;padding:5px 7px'>Highest cluster</th>"
        "<th style='text-align:right;padding:5px 7px'>High index</th><th style='text-align:left;padding:5px 7px'>Lowest cluster</th>"
        "<th style='text-align:right;padding:5px 7px'>Low index</th></tr>" + "".join(_rows) + "</table>"
    ))

if not EXEC_EVIDENCE_ONLY_JOB_INDEX.empty:
    _rows = []
    for r in EXEC_EVIDENCE_ONLY_JOB_INDEX.sort_values(["Job", "cluster"]).itertuples(index=False):
        _rows.append(
            "<tr style='border-bottom:1px solid #eee'>"
            f"<td style='padding:5px 7px;font-weight:650'>{escape(r.Job)}</td>"
            f"<td style='padding:5px 7px'>C{r.cluster} {escape(r.cluster_label)}</td>"
            f"<td style='padding:5px 7px;text-align:right'>{_idx_badge(r.index_vs_overall)}</td>"
            f"<td style='padding:5px 7px;text-align:right'>{r.measured_share:.0%}</td>"
            "</tr>"
        )
    display(HTML(
        "<div style='margin-top:14px;font-weight:700'>Evidence-only Jobs</div>"
        "<div style='font-size:11px;color:#666;margin-bottom:5px'>These do not receive Top-2 slots; the index compares mean raw behavioral evidence with overall.</div>"
        "<table style='border-collapse:collapse;width:100%;font-size:11px'>"
        "<tr style='background:#f8fafc'><th style='text-align:left;padding:5px 7px'>Job</th><th style='text-align:left;padding:5px 7px'>Cluster</th>"
        "<th style='text-align:right;padding:5px 7px'>Evidence index</th><th style='text-align:right;padding:5px 7px'>Measured</th></tr>"
        + "".join(_rows) + "</table>"
    ))

_title("I. Job combinations", "Top over-indexed Top-2 Job pairs by cluster; useful for deciding which Jobs might be sequenced or combined rather than treated independently.")
display(HTML(_job_pair_cards_html()))

_title("J. Core behavioral fingerprint in raw terms", "Same clustering ingredients, but reported as raw values and indexes instead of +/- SD unit scores.")
display(HTML(_metric_matrix("Core behavior")))

_title("K. Audience context", "Useful context for activation and treatment design; these variables did not define the cluster geometry.")
display(HTML(_metric_matrix("Audience context")))

# -----------------------------------------------------------------------------
# 12. One compact cluster-by-cluster readout for later DOCX synthesis
# -----------------------------------------------------------------------------

_title("L. Extended cluster cards", "Top over/under-indexes from each analytical layer, plus strongest Job tendencies.")

_layers_for_cards = [
    "Economic value", "Lifecycle + retention", "Digital product intensity",
    "Optional tip economics", "Choice + need affinity"
]

for cluster in sorted(EXEC_BASE["cluster"].unique()):
    n = int(EXEC_BASE["cluster"].eq(cluster).sum())
    share = n / len(EXEC_BASE)
    card_parts = [
        f"<div style='border:1px solid #d1d5db;border-radius:9px;padding:14px 16px;margin:12px 0 20px;background:white'>",
        f"<div style='font-size:20px;font-weight:750'>C{cluster} {escape(CLUSTER_LABELS.get(int(cluster), ''))}</div>",
        f"<div style='font-size:12px;color:#666;margin-bottom:9px'>{n:,} donors | {share:.1%} of clustered population</div>",
    ]

    for layer in _layers_for_cards:
        d = EXEC_PROFILE_LONG.loc[
            EXEC_PROFILE_LONG["cluster"].eq(cluster) & EXEC_PROFILE_LONG["layer"].eq(layer)
        ].copy()
        d = d.dropna(subset=["index_vs_overall"])
        if d.empty:
            continue
        d["distance_from_1"] = (d["index_vs_overall"] - 1).abs()
        d = d.sort_values("distance_from_1", ascending=False).head(4)
        card_parts.append(f"<div style='font-size:13px;font-weight:700;margin-top:9px'>{escape(layer)}</div>")
        card_parts.append("<table style='width:100%;font-size:11px;border-collapse:collapse'>")
        for _, r in d.iterrows():
            card_parts.append(
                "<tr style='border-bottom:1px solid #f0f0f0'>"
                f"<td style='padding:4px 6px'>{escape(r['metric'])}</td>"
                f"<td style='padding:4px 6px;text-align:right;font-weight:650'>{_fmt(r['cluster_value'], r['format'])}</td>"
                f"<td style='padding:4px 6px;text-align:right;color:#666'>overall {_fmt(r['overall_value'], r['format'])}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{_idx_badge(r['index_vs_overall'])}</td>"
                "</tr>"
            )
        card_parts.append("</table>")

    if not EXEC_JOB_INDEX_LONG.empty:
        top_col = f"top_{globals().get('JTBD_TOP_N', 2)}_share"
        jd = EXEC_JOB_INDEX_LONG.loc[EXEC_JOB_INDEX_LONG["cluster"].eq(cluster)].copy()
        jd = jd.sort_values(["index_vs_overall", top_col], ascending=[False, False]).head(4)
        card_parts.append("<div style='font-size:13px;font-weight:700;margin-top:9px'>Strongest Job tendencies</div>")
        card_parts.append("<table style='width:100%;font-size:11px;border-collapse:collapse'>")
        for _, r in jd.iterrows():
            card_parts.append(
                "<tr style='border-bottom:1px solid #f0f0f0'>"
                f"<td style='padding:4px 6px'>{escape(r['Job'])}</td>"
                f"<td style='padding:4px 6px;text-align:right'>{r[top_col]:.0%} Top-2</td>"
                f"<td style='padding:4px 6px;text-align:right'>{_idx_badge(r['index_vs_overall'])}</td>"
                "</tr>"
            )
        card_parts.append("</table>")

    card_parts.append("</div>")
    display(HTML("".join(card_parts)))

# -----------------------------------------------------------------------------
# 13. Objects preserved for the follow-up report
# -----------------------------------------------------------------------------

EXEC_MODEL_SPEC = pd.DataFrame([
    {
        "unit": unit,
        "unit_weight": float(globals().get("UNIT_WEIGHTS", {}).get(unit, 1.0)),
        "feature": feature,
        "direction": "+" if direction > 0 else "-",
    }
    for unit, spec in UNITS_NOW.items()
    for feature, direction in spec.items()
])

EXEC_ALL_OBJECTS = {
    "base": EXEC_BASE,
    "model_spec": EXEC_MODEL_SPEC,
    "availability_audit": EXEC_AVAILABILITY_AUDIT,
    "profile_long": EXEC_PROFILE_LONG,
    "revenue_concentration": EXEC_REVENUE_CONCENTRATION,
    "distributions": EXEC_DISTRIBUTIONS,
    "category_index_long": EXEC_CATEGORY_INDEX_LONG,
    "job_index_long": EXEC_JOB_INDEX_LONG,
    "job_portfolio": EXEC_JOB_PORTFOLIO,
    "job_coverage": EXEC_JOB_COVERAGE,
    "evidence_only_job_index": EXEC_EVIDENCE_ONLY_JOB_INDEX,
    "job_pair_index_long": EXEC_JOB_PAIR_INDEX_LONG,
}

print(
    "Created: EXEC_BASE, EXEC_MODEL_SPEC, EXEC_AVAILABILITY_AUDIT, "
    "EXEC_PROFILE_LONG, EXEC_REVENUE_CONCENTRATION, EXEC_DISTRIBUTIONS, "
    "EXEC_CATEGORY_INDEX_LONG, EXEC_JOB_INDEX_LONG, EXEC_JOB_PORTFOLIO, EXEC_JOB_COVERAGE, "
    "EXEC_EVIDENCE_ONLY_JOB_INDEX, EXEC_JOB_PAIR_INDEX_LONG, EXEC_ALL_OBJECTS"
)


Layer,Need,Status,Field / object,Coverage,Interpretation
Economic value,Observed lifetime revenue,AVAILABLE,lifetime_amount,100%,Available lifetime project-gift revenue; this is not predictive LTV.
Economic value,Predictive LTV,MISSING / FOLLOW-ON,-,-,Requires a true LTV/predicted future-value field or separate model.
Economic value,Current-window project revenue,AVAILABLE,gift_amount_24m,100%,Current notebook window is 24 months.
Economic value,Current-window all-giving revenue,AVAILABLE,grand_amount_24m,100%,Preferred if project + monthly dollars are available.
Lifecycle + retention,Current lifecycle state,AVAILABLE,is_continuing_donor_24m,100%,New/reactivated/continuing are current-window state descriptors.
Lifecycle + retention,Future retention / next-gift outcome,MISSING / FOLLOW-ON,-,-,Needs a subsequent observation window; should remain post-fit.
Digital product intensity,Site-active days,AVAILABLE,days_with_site_activity_24m,96%,Available if site source is observed.
Digital product intensity,True-session grain,AVAILABLE,site_unit_is_session,100%,"If 0, visit counts are donor-day rather than true sessions."
Digital product intensity,Project-page activity,AVAILABLE,project_page_visits_day_total_24m,96%,Available in current site feature family if loaded.
Digital product intensity,Unique projects viewed per session,MISSING / FOLLOW-ON,-,-,Not present in the current dictionary unless a newer feature build added it.


Metric,Overall,1Loyal Local Regulars,2Steady Explorers,3Bursty Project Solvers,4Need-Responsive Finishers,5Local Loyal Campaign Responders,6Distant Relationship Loyalists
24m project revenue per donor,$450,$3630.81x,$3650.81x,"$1,5513.45x OVER",$3530.78x UNDER,$3550.79x UNDER,$4000.89x
Previous equal-length-window project revenue per donorPrior window matches the configured current-window length,$41763% measured,$3350.80x63% measured,$2600.62x UNDER65% measured,"$1,4013.36x OVER74% measured",$3980.95x40% measured,$3610.87x69% measured,$3000.72x UNDER70% measured
24m all-giving revenue per donorProject + monthly if available,$453,$3670.81x,$3660.81x,"$1,5923.51x OVER",$3550.78x UNDER,$3550.78x UNDER,$4050.89x
"Lifetime project revenue per donorObserved lifetime revenue, not predictive LTV",$992,$8000.81x,$7250.73x UNDER,"$4,4214.46x OVER",$5870.59x UNDER,$8480.85x,$8650.87x
Lifetime project gifts,19.0,19.01.00x,15.00.79x UNDER,41.02.16x OVER,16.00.84x,16.00.84x,15.00.79x UNDER
Median project gift,$30,$250.83x,$280.92x,$812.71x OVER,$150.51x UNDER,$351.17x,$501.67x OVER
Major-gift donor share,0%,0%0.00x UNDER,0%0.00x UNDER,0%4.48x OVER,0%1.43x OVER,0%0.00x UNDER,0%0.00x UNDER


Revenue measure,Cluster,Donor share,Dollars,Revenue share,Revenue index
24m project revenue,C1 Loyal Local Regulars,22.5%,"$8,905,864",9.1%,0.41x UNDER
24m project revenue,C2 Steady Explorers,21.5%,"$8,315,976",8.5%,0.40x UNDER
24m project revenue,C3 Bursty Project Solvers,17.4%,"$46,606,493",47.7%,2.75x OVER
24m project revenue,C4 Need-Responsive Finishers,15.6%,"$22,763,445",23.3%,1.50x OVER
24m project revenue,C5 Local Loyal Campaign Responders,14.6%,"$7,413,127",7.6%,0.52x UNDER
24m project revenue,C6 Distant Relationship Loyalists,8.5%,"$3,603,987",3.7%,0.44x UNDER
24m all-giving revenue,C1 Loyal Local Regulars,22.5%,"$9,360,654",9.4%,0.42x UNDER
24m all-giving revenue,C2 Steady Explorers,21.5%,"$8,455,300",8.5%,0.40x UNDER
24m all-giving revenue,C3 Bursty Project Solvers,17.4%,"$47,243,825",47.7%,2.74x OVER
24m all-giving revenue,C4 Need-Responsive Finishers,15.6%,"$22,866,597",23.1%,1.48x OVER


Metric,Cluster,P25,Median,P75
24m project revenue per donor,C1 Loyal Local Regulars,$185,$363,$771
24m project revenue per donor,C2 Steady Explorers,$196,$365,$725
24m project revenue per donor,C3 Bursty Project Solvers,$723,"$1,551","$3,768"
24m project revenue per donor,C4 Need-Responsive Finishers,$163,$353,$882
24m project revenue per donor,C5 Local Loyal Campaign Responders,$181,$355,$758
24m project revenue per donor,C6 Distant Relationship Loyalists,$225,$400,$794
Lifetime project revenue per donor,C1 Loyal Local Regulars,$307,$800,"$2,231"
Lifetime project revenue per donor,C2 Steady Explorers,$323,$725,"$1,771"
Lifetime project revenue per donor,C3 Bursty Project Solvers,"$1,650","$4,421","$11,662"
Lifetime project revenue per donor,C4 Need-Responsive Finishers,$220,$587,"$1,925"


Metric,Overall,1Loyal Local Regulars,2Steady Explorers,3Bursty Project Solvers,4Need-Responsive Finishers,5Local Loyal Campaign Responders,6Distant Relationship Loyalists
Giving tenure,"1,802 days","1,964 days1.09x","1,585 days0.88x","2,578 days1.43x OVER",813 days0.45x UNDER,"1,815 days1.01x","2,030 days1.13x"
Days since last project giftLower = more recent,136 days,98 days0.72x UNDER,110 days0.81x,108 days0.79x UNDER,211 days1.55x OVER,199 days1.46x OVER,110 days0.81x
Active giving months,5.0,6.01.20x OVER,5.01.00x,6.01.20x OVER,3.00.60x UNDER,4.00.80x UNDER,5.01.00x
New-donor share,28%,28%1.00x,28%0.98x,19%0.67x UNDER,48%1.69x OVER,24%0.83x,21%0.74x UNDER
Reactivated-donor share,9%,8%0.96x,7%0.88x,7%0.84x,12%1.47x OVER,8%0.91x,9%1.04x
Continuing-donor share,63%,63%1.00x,65%1.02x,74%1.17x,40%0.63x UNDER,69%1.09x,70%1.11x
Current monthly-donor share,3%,5%1.92x OVER,1%0.52x UNDER,4%1.59x OVER,2%0.56x UNDER,0%0.07x UNDER,3%0.99x
Monthly active months,0.0,0.0n/a,0.0n/a,0.0n/a,0.0n/a,0.0n/a,0.0n/a
Longest monthly streak,20.05% measured,23.01.15x9% measured,11.00.55x UNDER4% measured,40.52.02x OVER8% measured,11.00.55x UNDER4% measured,8.00.40x UNDER2% measured,16.00.80x UNDER6% measured
Share of giving from monthly,1%,2%1.89x OVER,1%0.66x UNDER,1%1.37x OVER,1%0.57x UNDER,0%0.07x UNDER,1%1.13x


...

Major-gift donor share,0%,overall 0%,4.48x OVER
Lifetime project revenue per donor,"$4,421",overall $992,4.46x OVER
24m all-giving revenue per donor,"$1,592",overall $453,3.51x OVER
24m project revenue per donor,"$1,551",overall $450,3.45x OVER
Longest monthly streak,40.5,overall 20.0,2.02x OVER
Median monthly payment,$50,overall $25,2.00x OVER
Current monthly-donor share,4%,overall 3%,1.59x OVER
Giving tenure,"2,578 days","overall 1,802 days",1.43x OVER
Search visit share,18%,overall 13%,1.34x OVER
Gifts preceded by search,24%,overall 18%,1.32x OVER
Search visits,6.7,overall 5.5,1.21x OVER


Median project gift,$15,overall $30,0.51x UNDER
Major-gift donor share,0%,overall 0%,1.43x OVER
Lifetime project revenue per donor,$587,overall $992,0.59x UNDER
24m project revenue per donor,$353,overall $450,0.78x UNDER
New-donor share,48%,overall 28%,1.69x OVER
Days since last project gift,211 days,overall 136 days,1.55x OVER
Giving tenure,813 days,"overall 1,802 days",0.45x UNDER
Reactivated-donor share,12%,overall 9%,1.47x OVER
Project-page visits,4.0,overall 13.0,0.31x UNDER
Site-active days,6.0,overall 17.0,0.35x UNDER
Site visits,6.0,overall 17.0,0.35x UNDER


Major-gift donor share,0%,overall 0%,0.00x UNDER
24m all-giving revenue per donor,$355,overall $453,0.78x UNDER
24m project revenue per donor,$355,overall $450,0.79x UNDER
Median project gift,$35,overall $30,1.17x
Share of giving from monthly,0%,overall 1%,0.07x UNDER
Current monthly-donor share,0%,overall 3%,0.07x UNDER
Longest monthly streak,8.0,overall 20.0,0.40x UNDER
Days since last project gift,199 days,overall 136 days,1.46x OVER
Site-active days,32.0,overall 17.0,1.88x OVER
Site visits,32.0,overall 17.0,1.88x OVER
Search visits,9.8,overall 5.5,1.77x OVER


Major-gift donor share,0%,overall 0%,0.00x UNDER
Median project gift,$50,overall $30,1.67x OVER
Previous equal-length-window project revenue per donor,$300,overall $417,0.72x UNDER
Lifetime project gifts,15.0,overall 19.0,0.79x UNDER
New-donor share,21%,overall 28%,0.74x UNDER
Longest monthly streak,16.0,overall 20.0,0.80x UNDER
Days since last project gift,110 days,overall 136 days,0.81x
Share of giving from monthly,1%,overall 1%,1.13x
Search visit share,9%,overall 13%,0.69x UNDER
Gifts preceded by search,13%,overall 18%,0.70x UNDER
Search visits,4.5,overall 5.5,0.82x


Created: EXEC_BASE, EXEC_MODEL_SPEC, EXEC_AVAILABILITY_AUDIT, EXEC_PROFILE_LONG, EXEC_REVENUE_CONCENTRATION, EXEC_DISTRIBUTIONS, EXEC_CATEGORY_INDEX_LONG, EXEC_JOB_INDEX_LONG, EXEC_JOB_PORTFOLIO, EXEC_JOB_COVERAGE, EXEC_EVIDENCE_ONLY_JOB_INDEX, EXEC_JOB_PAIR_INDEX_LONG, EXEC_ALL_OBJECTS


In [14]:
# ============================================================================
# $10K+ DONORS + DAF ACTIVITY BY CLUSTER
# Paste after the clustering/profile cells.
# ============================================================================

import numpy as np
import pandas as pd
from IPython.display import HTML, display


# ----------------------------------------------------------------------------
# 1. Start with clustered donors
# ----------------------------------------------------------------------------

if "DONORS_WITH_CLUSTER" not in globals():
    raise NameError("DONORS_WITH_CLUSTER not found. Run the clustering cells first.")

x = DONORS_WITH_CLUSTER.copy()

if x.index.name != "donor_id" and "donor_id" in x.columns:
    x = x.set_index("donor_id")


# ----------------------------------------------------------------------------
# 2. Find the relevant fields
#
# Supports both the newer _24m names and legacy _12m names where the
# configured window itself is 24 months.
# ----------------------------------------------------------------------------

revenue_candidates = [
    "grand_amount_24m",   # project + monthly, preferred
    "grand_amount_12m",   # legacy suffix, may still represent 24m
    "gift_amount_24m",    # project gifts only, fallback
    "gift_amount_12m",
]

daf_candidates = [
    "share_gifts_daf_24m",
    "share_gifts_daf_12m",
]

needed_candidates = revenue_candidates + daf_candidates


# Pull missing fields from FEATURES_PATH if needed.
missing = [c for c in needed_candidates if c not in x.columns]

if (
    missing
    and "FEATURES_PATH" in globals()
    and FEATURES_PATH.exists()
):
    header = set(pd.read_csv(FEATURES_PATH, nrows=0).columns)

    loadable = [c for c in missing if c in header]

    if loadable:
        extra = (
            pd.read_csv(
                FEATURES_PATH,
                usecols=["donor_id"] + loadable,
            )
            .set_index("donor_id")
        )

        x = x.join(extra, how="left")


revenue_field = next(
    (c for c in revenue_candidates if c in x.columns),
    None,
)

daf_field = next(
    (c for c in daf_candidates if c in x.columns),
    None,
)


if revenue_field is None:
    raise KeyError(
        "Could not find a current-window revenue field. Tried: "
        + ", ".join(revenue_candidates)
    )

if daf_field is None:
    raise KeyError(
        "Could not find a current-window DAF field. Tried: "
        + ", ".join(daf_candidates)
    )


# ----------------------------------------------------------------------------
# 3. Create donor flags
# ----------------------------------------------------------------------------

revenue = pd.to_numeric(
    x[revenue_field],
    errors="coerce",
)

daf_share = pd.to_numeric(
    x[daf_field],
    errors="coerce",
)


x["gave_10k_plus_24m"] = revenue >= 10_000
x["any_daf_24m"] = daf_share > 0
x["gave_10k_plus_and_daf_24m"] = (
    x["gave_10k_plus_24m"]
    & x["any_daf_24m"]
)


# ----------------------------------------------------------------------------
# 4. Overall rates
# ----------------------------------------------------------------------------

overall_10k = x["gave_10k_plus_24m"].mean()
overall_daf = x["any_daf_24m"].mean()
overall_both = x["gave_10k_plus_and_daf_24m"].mean()


# ----------------------------------------------------------------------------
# 5. Cluster summary
# ----------------------------------------------------------------------------

rows = []

for cluster, g in x.groupby("cluster"):

    n = len(g)

    n_10k = int(g["gave_10k_plus_24m"].sum())
    n_daf = int(g["any_daf_24m"].sum())
    n_both = int(g["gave_10k_plus_and_daf_24m"].sum())

    pct_10k = n_10k / n
    pct_daf = n_daf / n
    pct_both = n_both / n

    rows.append({
        "Cluster": int(cluster),
        "Donors": n,

        "$10K+ donors": n_10k,
        "$10K+ share": pct_10k,
        "$10K+ index": (
            pct_10k / overall_10k
            if overall_10k > 0 else np.nan
        ),

        "Any DAF donors": n_daf,
        "Any DAF share": pct_daf,
        "DAF index": (
            pct_daf / overall_daf
            if overall_daf > 0 else np.nan
        ),

        "$10K+ & DAF": n_both,
        "Both share": pct_both,
        "Both index": (
            pct_both / overall_both
            if overall_both > 0 else np.nan
        ),
    })


HIGH_VALUE_DAF_BY_CLUSTER = pd.DataFrame(rows).sort_values("Cluster")


# Overall row
overall_row = pd.DataFrame([{
    "Cluster": "OVERALL",
    "Donors": len(x),

    "$10K+ donors": int(x["gave_10k_plus_24m"].sum()),
    "$10K+ share": overall_10k,
    "$10K+ index": 1.0,

    "Any DAF donors": int(x["any_daf_24m"].sum()),
    "Any DAF share": overall_daf,
    "DAF index": 1.0,

    "$10K+ & DAF": int(x["gave_10k_plus_and_daf_24m"].sum()),
    "Both share": overall_both,
    "Both index": 1.0,
}])


display_table = pd.concat(
    [HIGH_VALUE_DAF_BY_CLUSTER, overall_row],
    ignore_index=True,
)


# ----------------------------------------------------------------------------
# 6. Easy-to-read HTML output
# ----------------------------------------------------------------------------

display(
    HTML(
        f"""
        <div style="
            border:1px solid #d1d5db;
            border-radius:8px;
            padding:12px 14px;
            margin:8px 0 14px 0;
            background:#f8fafc;
            font-size:13px;
        ">
            <b>Fields used</b><br>
            $10K threshold: <code>{revenue_field}</code><br>
            DAF activity: <code>{daf_field}</code><br><br>

            <b>Important:</b> DAF means any observed DAF project gift in the
            current configured 24-month window. The current feature build does
            not contain a lifetime-ever DAF flag.
        </div>
        """
    )
)


display(
    display_table.style
    .format({
        "Donors": "{:,.0f}",
        "$10K+ donors": "{:,.0f}",
        "$10K+ share": "{:.1%}",
        "$10K+ index": "{:.2f}x",
        "Any DAF donors": "{:,.0f}",
        "Any DAF share": "{:.1%}",
        "DAF index": "{:.2f}x",
        "$10K+ & DAF": "{:,.0f}",
        "Both share": "{:.1%}",
        "Both index": "{:.2f}x",
    })
    .hide(axis="index")
    .set_caption(
        "$10K+ giving and DAF activity by behavioral cluster"
    )
)


print("Saved as HIGH_VALUE_DAF_BY_CLUSTER")

Cluster,Donors,$10K+ donors,$10K+ share,$10K+ index,Any DAF donors,Any DAF share,DAF index,$10K+ & DAF,Both share,Both index
1,"12,265",37,0.3%,0.13x,99,0.8%,0.72x,13,0.1%,0.23x
2,"11,731",47,0.4%,0.17x,28,0.2%,0.21x,6,0.1%,0.11x
3,"9,492",848,8.9%,3.88x,339,3.6%,3.19x,183,1.9%,4.20x
4,"8,513",219,2.6%,1.12x,119,1.4%,1.25x,43,0.5%,1.10x
5,"7,994",86,1.1%,0.47x,10,0.1%,0.11x,6,0.1%,0.16x
6,"4,636",21,0.5%,0.20x,17,0.4%,0.33x,0,0.0%,0.00x
OVERALL,"54,631","1,258",2.3%,1.00x,612,1.1%,1.00x,251,0.5%,1.00x


Saved as HIGH_VALUE_DAF_BY_CLUSTER


In [15]:
# ============================================================================
# OPTIONAL-DONATION PROFIT GAP TO 15%
# Donor-level calculation -> aggregate by cluster
# Deep dive: $10K+ donors vs everyone else
# ============================================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import HTML, display


# ----------------------------------------------------------------------------
# 0. Settings
# ----------------------------------------------------------------------------

TARGET_OPTIONAL_RATE = 0.15
LARGE_DONOR_THRESHOLD = 10_000

CLUSTER_LABELS_LOCAL = {
    1: "Loyal Local Regulars",
    2: "Steady Explorers",
    3: "Bursty Project Solvers",
    4: "Need-Responsive Finishers",
    5: "Local Loyal Campaign Responders",
    6: "Distant Relationship Loyalists",
}


# ----------------------------------------------------------------------------
# 1. Start with clustered donors
# ----------------------------------------------------------------------------

if "DONORS_WITH_CLUSTER" not in globals():
    raise NameError("DONORS_WITH_CLUSTER not found. Run the clustering cells first.")

x = DONORS_WITH_CLUSTER.copy()

if x.index.name != "donor_id" and "donor_id" in x.columns:
    x = x.set_index("donor_id")


# ----------------------------------------------------------------------------
# 2. Load needed fields if they are not already present
# ----------------------------------------------------------------------------

candidate_fields = [
    # optional donation rate
    "avg_optional_donation_rate_24m",
    "avg_optional_donation_rate_12m",

    # direct green-dollar fields, if upstream happens to contain one
    "green_revenue_24m",
    "green_revenue_12m",
    "gift_amount_green_24m",
    "gift_amount_green_12m",
    "green_amount_24m",
    "green_amount_12m",

    # otherwise derive green dollars
    "gift_amount_24m",
    "gift_amount_12m",
    "share_amount_green_optin_24m",
    "share_amount_green_optin_12m",
    "share_amount_green_24m",
    "share_amount_green_12m",

    # $10K+ donor definition: project + monthly preferred
    "grand_amount_24m",
    "grand_amount_12m",
]

missing = [c for c in candidate_fields if c not in x.columns]

if (
    missing
    and "FEATURES_PATH" in globals()
    and Path(FEATURES_PATH).exists()
):
    header = set(pd.read_csv(FEATURES_PATH, nrows=0).columns)

    loadable = [c for c in missing if c in header]

    if loadable:
        extra = (
            pd.read_csv(
                FEATURES_PATH,
                usecols=["donor_id"] + loadable,
            )
            .set_index("donor_id")
        )

        x = x.join(extra, how="left")


def first_existing(candidates):
    return next((c for c in candidates if c in x.columns), None)


# ----------------------------------------------------------------------------
# 3. Resolve fields
# ----------------------------------------------------------------------------

optional_rate_field = first_existing([
    "avg_optional_donation_rate_24m",
    "avg_optional_donation_rate_12m",
])

if optional_rate_field is None:
    raise KeyError("Could not find avg_optional_donation_rate field.")


large_donor_revenue_field = first_existing([
    "grand_amount_24m",
    "grand_amount_12m",
    "gift_amount_24m",
    "gift_amount_12m",
])

if large_donor_revenue_field is None:
    raise KeyError("Could not find a 24m revenue field for the $10K+ donor flag.")


# ----------------------------------------------------------------------------
# 4. Resolve / derive GREEN revenue at donor level
# ----------------------------------------------------------------------------

direct_green_field = first_existing([
    "green_revenue_24m",
    "green_revenue_12m",
    "gift_amount_green_24m",
    "gift_amount_green_12m",
    "green_amount_24m",
    "green_amount_12m",
])

green_derivation_note = ""

if direct_green_field is not None:

    x["_green_revenue"] = pd.to_numeric(
        x[direct_green_field],
        errors="coerce"
    )

    green_revenue_source = direct_green_field

else:

    project_revenue_field = first_existing([
        "gift_amount_24m",
        "gift_amount_12m",
    ])

    green_share_field = first_existing([
        # Preferred: explicitly described upstream as
        # Green payment amount / total project-gift amount.
        "share_amount_green_optin_24m",
        "share_amount_green_optin_12m",

        # fallback if the preferred field is not available
        "share_amount_green_24m",
        "share_amount_green_12m",
    ])

    if project_revenue_field is None or green_share_field is None:
        raise KeyError(
            "Could not derive Green revenue. Need either a direct Green-dollar "
            "field or both gift_amount_* and share_amount_green_*."
        )

    project_revenue = pd.to_numeric(
        x[project_revenue_field],
        errors="coerce"
    )

    green_share = pd.to_numeric(
        x[green_share_field],
        errors="coerce"
    )

    x["_green_revenue"] = project_revenue * green_share

    green_revenue_source = (
        f"{project_revenue_field} × {green_share_field}"
    )

    green_derivation_note = (
        "Green revenue was derived from total project-gift dollars × "
        "Green dollar share."
    )


# ----------------------------------------------------------------------------
# 5. Normalize optional-rate units
#
# Some source fields may be stored as 0.15 while others may be stored as 15.
# Detect this automatically.
# ----------------------------------------------------------------------------

rate_raw = pd.to_numeric(
    x[optional_rate_field],
    errors="coerce"
)

non_null_rate = rate_raw.dropna()

if non_null_rate.empty:
    raise ValueError("Optional-donation-rate field contains no usable values.")

# If normal values are clearly above 1, treat source as percent units.
if non_null_rate.quantile(0.95) > 1:
    x["_optional_rate"] = rate_raw / 100.0
    rate_unit_note = (
        f"{optional_rate_field} appeared to use percent units and was divided by 100."
    )
else:
    x["_optional_rate"] = rate_raw
    rate_unit_note = (
        f"{optional_rate_field} appeared to use decimal rate units."
    )


# Safety checks
bad_rate = (
    x["_optional_rate"].notna()
    & (
        (x["_optional_rate"] < 0)
        | (x["_optional_rate"] > 1)
    )
)

if bad_rate.any():
    raise ValueError(
        f"{bad_rate.sum():,} donors have optional rates outside 0–100% "
        "after unit normalization. Inspect the source field before using results."
    )


# ----------------------------------------------------------------------------
# 6. DONOR-LEVEL economics
#
# "Profit lost" here means estimated optional-donation dollars below a
# hypothetical 15% rate.
#
# actual optional $ estimate:
#     green revenue × observed average optional rate
#
# potential optional $ at 15%:
#     green revenue × 15%
#
# lost $:
#     green revenue × max(15% - observed rate, 0)
#
# Donors already at or above 15% get zero lost dollars.
# ----------------------------------------------------------------------------

x["_optional_rate_gap"] = np.maximum(
    TARGET_OPTIONAL_RATE - x["_optional_rate"],
    0
)

x["_estimated_optional_dollars_actual"] = (
    x["_green_revenue"]
    * x["_optional_rate"]
)

x["_estimated_optional_dollars_at_15pct"] = (
    x["_green_revenue"]
    * TARGET_OPTIONAL_RATE
)

x["_absolute_profit_lost"] = (
    x["_green_revenue"]
    * x["_optional_rate_gap"]
)


# ----------------------------------------------------------------------------
# 7. $10K+ donor flag
# ----------------------------------------------------------------------------

large_donor_revenue = pd.to_numeric(
    x[large_donor_revenue_field],
    errors="coerce"
)

x["_large_donor_10k"] = (
    large_donor_revenue >= LARGE_DONOR_THRESHOLD
)

x["_donor_group"] = np.where(
    x["_large_donor_10k"],
    "$10K+ donors",
    "All other donors"
)


# Only donors with usable inputs enter the economics calculation.
x["_profit_calc_usable"] = (
    x["_green_revenue"].notna()
    & x["_optional_rate"].notna()
)


# ----------------------------------------------------------------------------
# 8. Aggregate helper
# ----------------------------------------------------------------------------

def summarize(g):

    usable = g[g["_profit_calc_usable"]].copy()

    green_rev = usable["_green_revenue"].sum()
    actual_optional = usable["_estimated_optional_dollars_actual"].sum()
    lost = usable["_absolute_profit_lost"].sum()

    # Dollar-weighted optional rate is more economically meaningful here
    # than simply averaging donor-level rates.
    weighted_rate = (
        actual_optional / green_rev
        if green_rev > 0 else np.nan
    )

    lost_pct_green = (
        lost / green_rev
        if green_rev > 0 else np.nan
    )

    return pd.Series({
        "Donors": len(g),
        "Usable donors": len(usable),
        "Coverage": len(usable) / len(g) if len(g) else np.nan,
        "Green revenue": green_rev,
        "Estimated optional $ received": actual_optional,
        "Dollar-weighted optional rate": weighted_rate,
        "Absolute $ profit lost": lost,
        "Lost $ / Green revenue": lost_pct_green,
    })


# ----------------------------------------------------------------------------
# 9. Main cluster table
# ----------------------------------------------------------------------------

cluster_summary = (
    x.groupby("cluster", observed=True)
    .apply(summarize)
    .reset_index()
)

cluster_summary["Cluster name"] = (
    cluster_summary["cluster"]
    .map(CLUSTER_LABELS_LOCAL)
)

cluster_summary["Share of total $ lost"] = (
    cluster_summary["Absolute $ profit lost"]
    / cluster_summary["Absolute $ profit lost"].sum()
)


# ----------------------------------------------------------------------------
# 10. Deep dive: $10K+ vs everyone else within each cluster
# ----------------------------------------------------------------------------

deep_dive = (
    x.groupby(
        ["cluster", "_donor_group"],
        observed=True
    )
    .apply(summarize)
    .reset_index()
    .rename(columns={"_donor_group": "Donor group"})
)

deep_dive["Cluster name"] = (
    deep_dive["cluster"]
    .map(CLUSTER_LABELS_LOCAL)
)


# Add within-cluster contribution to loss.
cluster_loss_lookup = (
    deep_dive
    .groupby("cluster")["Absolute $ profit lost"]
    .sum()
)

deep_dive["Share of cluster $ lost"] = (
    deep_dive.apply(
        lambda r:
            r["Absolute $ profit lost"]
            / cluster_loss_lookup.loc[r["cluster"]]
            if cluster_loss_lookup.loc[r["cluster"]] > 0
            else np.nan,
        axis=1
    )
)


# ----------------------------------------------------------------------------
# 11. Wide executive table:
#     total lost + amount attributable to large donors vs everyone else
# ----------------------------------------------------------------------------

loss_wide = (
    deep_dive
    .pivot(
        index=["cluster", "Cluster name"],
        columns="Donor group",
        values="Absolute $ profit lost"
    )
    .reset_index()
)

for c in ["$10K+ donors", "All other donors"]:
    if c not in loss_wide.columns:
        loss_wide[c] = 0.0

loss_wide = loss_wide.rename(columns={
    "$10K+ donors": "$ lost — $10K+ donors",
    "All other donors": "$ lost — all others",
})

loss_wide["Total $ lost"] = (
    loss_wide["$ lost — $10K+ donors"]
    + loss_wide["$ lost — all others"]
)

loss_wide["% of loss from $10K+ donors"] = np.where(
    loss_wide["Total $ lost"] > 0,
    loss_wide["$ lost — $10K+ donors"]
    / loss_wide["Total $ lost"],
    np.nan
)


# Merge useful cluster context
loss_wide = loss_wide.merge(
    cluster_summary[
        [
            "cluster",
            "Green revenue",
            "Dollar-weighted optional rate",
            "Share of total $ lost",
        ]
    ],
    on="cluster",
    how="left"
)


# ----------------------------------------------------------------------------
# 12. Overall totals
# ----------------------------------------------------------------------------

overall = summarize(x)

overall_large_loss = x.loc[
    x["_large_donor_10k"] & x["_profit_calc_usable"],
    "_absolute_profit_lost"
].sum()

overall_other_loss = x.loc[
    (~x["_large_donor_10k"]) & x["_profit_calc_usable"],
    "_absolute_profit_lost"
].sum()

overall_total_loss = overall_large_loss + overall_other_loss

# Total optional-donation opportunity if all usable Green revenue carried 15%
overall_total_oppty = (
    x.loc[x["_profit_calc_usable"], "_green_revenue"].sum()
    * TARGET_OPTIONAL_RATE
)

# ----------------------------------------------------------------------------
# 13. Save useful objects
# ----------------------------------------------------------------------------

OPTIONAL_PROFIT_GAP_DONOR_LEVEL = x[
    [
        "cluster",
        "_green_revenue",
        "_optional_rate",
        "_optional_rate_gap",
        "_estimated_optional_dollars_actual",
        "_estimated_optional_dollars_at_15pct",
        "_absolute_profit_lost",
        "_large_donor_10k",
        "_profit_calc_usable",
    ]
].copy()

OPTIONAL_PROFIT_GAP_BY_CLUSTER = cluster_summary.copy()
OPTIONAL_PROFIT_GAP_DEEP_DIVE = deep_dive.copy()
OPTIONAL_PROFIT_GAP_EXEC = loss_wide.copy()


# ----------------------------------------------------------------------------
# 14. Easy-to-read HTML output
# ----------------------------------------------------------------------------

display(
    HTML(
        f"""
        <div style="
            border:1px solid #d1d5db;
            border-radius:9px;
            padding:13px 15px;
            margin:8px 0 16px 0;
            background:#f8fafc;
            font-size:13px;
            line-height:1.45;
        ">
            <div style="font-size:17px;font-weight:750;margin-bottom:6px">
                Optional-donation profit gap to 15%
            </div>

            <b>Donor-level formula:</b><br>
            Green revenue × max(15% − donor average optional rate, 0)<br><br>

            <b>Green revenue:</b>
            <code>{green_revenue_source}</code><br>

            <b>Large donor:</b>
            ≥ ${LARGE_DONOR_THRESHOLD:,.0f} using
            <code>{large_donor_revenue_field}</code><br>

            <b>Optional-rate field:</b>
            <code>{optional_rate_field}</code><br><br>

            {green_derivation_note}<br>
            {rate_unit_note}
        </div>
        """
    )
)


display(
    HTML(
        f"""
        <div style="
            display:flex;
            gap:12px;
            flex-wrap:wrap;
            margin:6px 0 18px 0;
        ">
            <div style="border:1px solid #ddd;border-radius:8px;padding:10px 14px">
                <div style="font-size:11px;color:#666">TOTAL EST. PROFIT OPPTY</div>
                <div style="font-size:23px;font-weight:750">
                    ${overall_total_oppty:,.0f}
                </div>
                <div style="font-size:11px;color:#666">
                    Green revenue × 15%
                </div>
            </div>

            <div style="border:1px solid #ddd;border-radius:8px;padding:10px 14px">
                <div style="font-size:11px;color:#666">TOTAL EST. PROFIT LOST</div>
                <div style="font-size:23px;font-weight:750">
                    ${overall_total_loss:,.0f}
                </div>
            </div>

            <div style="border:1px solid #ddd;border-radius:8px;padding:10px 14px">
                <div style="font-size:11px;color:#666">FROM $10K+ DONORS</div>
                <div style="font-size:23px;font-weight:750">
                    ${overall_large_loss:,.0f}
                </div>
                <div style="font-size:11px;color:#666">
                    {(overall_large_loss / overall_total_loss if overall_total_loss else np.nan):.1%}
                    of total loss
                </div>
            </div>

            <div style="border:1px solid #ddd;border-radius:8px;padding:10px 14px">
                <div style="font-size:11px;color:#666">FROM ALL OTHER DONORS</div>
                <div style="font-size:23px;font-weight:750">
                    ${overall_other_loss:,.0f}
                </div>
                <div style="font-size:11px;color:#666">
                    {(overall_other_loss / overall_total_loss if overall_total_loss else np.nan):.1%}
                    of total loss
                </div>
            </div>
        </div>
        """
    )
)


display(
    loss_wide[
        [
            "cluster",
            "Cluster name",
            "Green revenue",
            "Dollar-weighted optional rate",
            "Total $ lost",
            "Share of total $ lost",
            "$ lost — $10K+ donors",
            "$ lost — all others",
            "% of loss from $10K+ donors",
        ]
    ]
    .style
    .format({
        "Green revenue": "${:,.0f}",
        "Dollar-weighted optional rate": "{:.1%}",
        "Total $ lost": "${:,.0f}",
        "Share of total $ lost": "{:.1%}",
        "$ lost — $10K+ donors": "${:,.0f}",
        "$ lost — all others": "${:,.0f}",
        "% of loss from $10K+ donors": "{:.1%}",
    })
    .hide(axis="index")
    .set_caption(
        "Estimated optional-donation profit lost by cluster"
    )
)


display(
    deep_dive[
        [
            "cluster",
            "Cluster name",
            "Donor group",
            "Donors",
            "Usable donors",
            "Coverage",
            "Green revenue",
            "Dollar-weighted optional rate",
            "Absolute $ profit lost",
            "Share of cluster $ lost",
        ]
    ]
    .style
    .format({
        "Donors": "{:,.0f}",
        "Usable donors": "{:,.0f}",
        "Coverage": "{:.1%}",
        "Green revenue": "${:,.0f}",
        "Dollar-weighted optional rate": "{:.1%}",
        "Absolute $ profit lost": "${:,.0f}",
        "Share of cluster $ lost": "{:.1%}",
    })
    .hide(axis="index")
    .set_caption(
        "$10K+ donor deep dive within each cluster"
    )
)


print(
    "Saved objects:",
    "OPTIONAL_PROFIT_GAP_DONOR_LEVEL,",
    "OPTIONAL_PROFIT_GAP_BY_CLUSTER,",
    "OPTIONAL_PROFIT_GAP_DEEP_DIVE,",
    "OPTIONAL_PROFIT_GAP_EXEC"
)

cluster,Cluster name,Green revenue,Dollar-weighted optional rate,Total $ lost,Share of total $ lost,$ lost — $10K+ donors,$ lost — all others,% of loss from $10K+ donors
1,Loyal Local Regulars,"$6,486,589",12.1%,"$184,944",4.5%,"$11,425","$173,519",6.2%
2,Steady Explorers,"$5,563,530",10.1%,"$271,944",6.7%,"$26,952","$244,992",9.9%
3,Bursty Project Solvers,"$37,563,337",10.0%,"$1,886,660",46.2%,"$1,256,972","$629,689",66.6%
4,Need-Responsive Finishers,"$16,926,397",7.5%,"$1,264,565",31.0%,"$889,823","$374,742",70.4%
5,Local Loyal Campaign Responders,"$5,211,180",7.9%,"$369,422",9.0%,"$108,594","$260,828",29.4%
6,Distant Relationship Loyalists,"$2,726,493",11.1%,"$105,971",2.6%,"$13,591","$92,380",12.8%


cluster,Cluster name,Donor group,Donors,Usable donors,Coverage,Green revenue,Dollar-weighted optional rate,Absolute $ profit lost,Share of cluster $ lost
1,Loyal Local Regulars,$10K+ donors,37,34,91.9%,"$363,109",11.9%,"$11,425",6.2%
1,Loyal Local Regulars,All other donors,"12,228","12,199",99.8%,"$6,123,479",12.2%,"$173,519",93.8%
2,Steady Explorers,$10K+ donors,47,47,100.0%,"$451,232",9.0%,"$26,952",9.9%
2,Steady Explorers,All other donors,"11,684","11,676",99.9%,"$5,112,297",10.2%,"$244,992",90.1%
3,Bursty Project Solvers,$10K+ donors,848,848,100.0%,"$21,614,848",9.2%,"$1,256,972",66.6%
3,Bursty Project Solvers,All other donors,"8,644","8,644",100.0%,"$15,948,489",11.1%,"$629,689",33.4%
4,Need-Responsive Finishers,$10K+ donors,219,217,99.1%,"$11,960,111",7.6%,"$889,823",70.4%
4,Need-Responsive Finishers,All other donors,"8,294","8,206",98.9%,"$4,966,285",7.5%,"$374,742",29.6%
5,Local Loyal Campaign Responders,$10K+ donors,86,86,100.0%,"$1,229,019",6.2%,"$108,594",29.4%
5,Local Loyal Campaign Responders,All other donors,"7,908","7,907",100.0%,"$3,982,161",8.5%,"$260,828",70.6%


Saved objects: OPTIONAL_PROFIT_GAP_DONOR_LEVEL, OPTIONAL_PROFIT_GAP_BY_CLUSTER, OPTIONAL_PROFIT_GAP_DEEP_DIVE, OPTIONAL_PROFIT_GAP_EXEC
